# Plotting notebook for robochem data
i was told that the way that the robochem plots stuff when running is not very user friendly and it doesn't look that good. So here's a Notebook that plots the DF of results in as many plots as possible with as much color as possible and also does some extra data analsyis. Have funsies!

## Instructions:

there is not much you should be doing here, but, if you see the python code: 
```python
### Change here: what to change
function...
### stop changing

```
It means that you should be changing something in the line after the ```### Change here:``` usually this is a path or a name or a label or smth like this.


### First step data loading and extra information:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.stats import pearsonr, spearmanr, mode
from itertools import combinations
from scipy.spatial import ConvexHull
from pypalettes import load_cmap
from typing import List, Dict, Any, Optional
import os
import random
import math
from sklearn.preprocessing import LabelEncoder
import plotly.graph_objects as go
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def prettify_column_names(col_string: str) -> str:
    """
    Pretty-fies a column name and appends units:
      - Replaces "_" with " ", title-cases
      - "Conc" ⇒ "Concentration"
      - If it's a concentration:
          • "Limiting Reagent" ⇒ [mM]

          • everything else ⇒ [equiv]
      - If it mentions "light", "yield", or "conversion" ⇒ [%]
      - If "temperature" ⇒ [°C]
      - If "residence time" ⇒ [s]
      - Otherwise ⇒ [-]
    """
    # base name
    # name = col_string.replace("_", " ")
    name = col_string
    name = name.replace("conc", "Concentration")

    # pick unit
    lower = name.lower()
    if "concentration" in lower:
        if "limiting reagent" in lower:
            unit = " [mM]"
        elif '*ctmp*' in lower:
            unit = " [mol%]"
            name = name.replace('*CTMP*', '')
        else:
            unit = " [equiv]"
    elif any(k in lower for k in ("light", "yield", "conversion")):
        unit = " [%]"
    elif "temperature" in lower:
        unit = " [°C]"
    elif "residence time" in lower:
        unit = " [s]"
    elif 'integral' in lower:
        unit = " [-]"
    elif 'excess' in lower:
        unit = " [%]"
    elif 'ratio' in lower:
        unit = " [-]"
    else:
        unit = ""

    if 'variance' in lower:
        name_parts = name.split(" ")
        return name_parts[0]+ ' ' + name_parts[1] + unit+ ' ' + name_parts[2]
    return f"{name}{unit}"

def sanitize_chemical_names(name: str) -> str:
    """
    Converts a chemical name string into a matplotlib-compatible LaTeX-style string.
    - '^X' becomes superscript: '$X^{X}$'
    - '_X' becomes subscript: '$X_{X}$'
    - All other characters are preserved as-is.

    Useful for making pretty axis tick labels in matplotlib.

    Args:
        name (str): Raw chemical name with markup hints.

    Returns:
        str: Sanitized string with LaTeX-style mathtext.
    """
    result = []
    i = 0
    while i < len(name):
        if name[i] == '^' and i + 1 < len(name):
            result.append(f"^{{{name[i+1]}}}")
            i += 2
        elif name[i] == '_' and i + 1 < len(name):
            result.append(f"_{{{name[i+1]}}}")
            i += 2
        elif name[i]== '*' and i+1 <len(name):
            result.append(f",")
            i += 1
        else:
            result.append(name[i])
            i += 1
    return f"${''.join(result)}$"

### Change here: Find a qualitative cmap for coloring (discrete values)
default_qualitative_cmap = 'kyogre'
### Change here: Find a sequential (perceptually uniform) cmap for coloring (continuous values)
default_sequential_cmap = 'enara'
### Change Here: finda a secondary sequential cmap for coloring (continuous values)
default_sequential_cmap2 = 'Emrld'
### stop changing here

print("See how your plots_data1 will feel like")
print("Qualitative cmap")
cmap = load_cmap(default_qualitative_cmap)
cmap

In [ ]:

#turn off the grid and background of matplotlub
plt.rcParams.update(plt.rcParamsDefault)


In [ ]:
print("Sequential cmap")
cmap2 = load_cmap(default_sequential_cmap, cmap_type='continuous')
cmap2

In [ ]:
print("Secondary Sequential cmap")
cmap3 = load_cmap(default_sequential_cmap2, cmap_type='continuous')
cmap3

In [ ]:
### Change here: put the path to the csv file
input_file = 'data/Experiment/DataFinal.csv'
### Change here: put the target variables
targets = ['yield']
### Change here: put the target variables that are both targets and variables
target_vars = ['Time']
### Change here: put the target variables that are to be minimised
minimise = ['Time']
### Change here: put to True to save data:
save_plots = True
### Change here: categorical numericals, add columns that although are numerical, they should be treated as categorical
categorical_numericals = []
### Change here: add exploration limit and optimum value
exploration_limit = 9
optimum_run= 23
sideprod_start = None
optimum_run_2 = None
### stop changing here



plot_saving_path = os.path.join(os.path.dirname(input_file), 'plots_data_final_8')

if not os.path.exists(plot_saving_path):
    os.makedirs(plot_saving_path)
if save_plots == False:
    plot_saving_path = None
data = pd.read_csv(input_file, header = 0)
data_copy = data.copy(deep=True)

# data cleaning, removinf dead/failed status and extra columns
if 'status' in data.columns:
    data = data[data['status'] == 'finished']

# save the predicted values in a separata dataframe and drop them from the main dataframe
if any([f"{tar}_predicted" in data.columns for tar in targets]):
    predicted_values = data[[f"{tar}_predicted" for tar in targets]]
    data.drop([f"{tar}_predicted" for tar in targets], axis=1, inplace=True, errors='ignore')

if any(["predicted" in column for column in data.columns]):
    predicted_values_2 = data[[column for column in data.columns if 'predicted' in column]]
    data.drop([column for column in data.columns if 'predicted' in column], axis=1, inplace = True, errors = 'ignore')

# if there is a light intensity column bin it in [0,25,


data.drop(['status', 'vial_idx', 'vial idx', 'run_index','run index', 'file_save_name', 'file save name', 'attempts', 'Ignore 1', 'Ignore 2', 'run'], axis=1, inplace=True, errors='ignore')
extra_ignores = ['Starting Material','Starting Material conc','Coupling Partner']
data.drop(extra_ignores, axis=1, inplace=True, errors='ignore')
# drop any column that has a single unique value:
data = data.loc[:, data.apply(pd.Series.nunique) > 1]

for col in data.columns:
    if '*CTMP*' in col:
        data[col] *= 100


data.columns = data.columns.map(prettify_column_names)
targets = [prettify_column_names(x) for x in targets]
target_vars = [prettify_column_names(x) for x in target_vars]
minimise = [prettify_column_names(x) for x in minimise]


print('data loaded successfully')
def categorise_columns(df: pd.DataFrame,
                       targets: List[str],
                       target_vars: List[str],
                       categorical_numericals: List[str] = None
                      ) -> Dict[str, List[str]]:
    """
    Categorise columns of a dataframe into numerical, categorical, and target columns.

    Parameters:
    df: pd.DataFrame - The DataFrame to categorize.
    targets: List[str] - List of column names that represent the target variables.
    target_vars: List[str] - Subset of target variables to be included as features.
    categorical_numericals: List[str] - List of columns with numerical values to treat as categorical.

    Returns:
    Dict[str, List[str]] - A dictionary with keys 'numerical', 'categorical', and 'target'
                           mapping to lists of column names.
    """

    # Determine which target columns should be held aside (not used as features)
    # Only include targets that are not in target_vars (i.e., not to be used as features)
    target_columns = [col for col in targets if col in df.columns and col not in target_vars]

    # Identify candidate numerical columns: columns with numeric dtypes and not reserved as target.
    numerical_columns = [
        col for col in df.select_dtypes(include=[np.number]).columns
        if col not in target_columns# and 'Light' not in col)
    ]

    # Identify candidate categorical columns: columns with object type and not reserved as target.
    categorical_columns = [
        col for col in df.select_dtypes(include=['object']).columns
        if col not in target_columns
    ]
    # if 'Light Intensity [%]' in df.columns:
    #     categorical_numericals.append('Light Intensity [%]')

    # Remove any categorical column that has only a single unique value
    categorical_columns = [col for col in categorical_columns if df[col].nunique() > 1]

    # Optionally, if certain numerical columns should be treated as categorical,
    # add them to categorical_columns and remove from numerical_columns.
    if categorical_numericals is not None:
        # Ensure these columns exist in the DataFrame
        valid_cat_num = [col for col in categorical_numericals if col in df.columns]
        categorical_columns = list(set(categorical_columns + valid_cat_num))
        numerical_columns = [col for col in numerical_columns if col not in valid_cat_num]

    return {
        'numerical': numerical_columns,
        'categorical': categorical_columns,
        'target': target_columns
    }

colum_types = categorise_columns(data, targets, target_vars, categorical_numericals = categorical_numericals)


print('columns categorised successfully')
print(data.columns)
print('numerical columns:', colum_types['numerical'])
print('categorical columns:', colum_types['categorical'])

# cleaning chemical names:
for col in colum_types['categorical']:
    print(f'Sanitizing chemical names in column: {col}')
    data[col] = data[col].astype(str).apply(sanitize_chemical_names)

# if needed split in smaller datasets§

# data_1 = data.iloc[:31,:]
# data_2 = data.iloc[31:,:]

# Plotting 1: Targets, and hypervolumes

The first plots that we present are:
- Each target vs the run number by themselves or both in the same graph after normalisation, 
- If multiple targets are available we also plot them against each other and we find the pareto front (up to 3d after we go by combinations)
- If multiple targets we plot the hypervolume over run number

In [ ]:
def plot_targets_vs_run_number(data: pd.DataFrame, targets: List[str], plot_saving_path: str = None, data_normalised = None, rolling_window : int = 5, **kwargs)-> None:
    '''
    Plots each target vs the run number, first as an overlayed plot with normalized data, and then each target
    individually with a shaded area representing the rolling median and percentile bands.

    Parameters:
    data: pd.DataFrame - The data to plot.
    targets: List[str] - List of column names of targets.
    plot_saving_path: str, optional - Path to save the plot (if None, no plot is saved).
    data_normalised: pd.DataFrame, optional - Buchwald Mapping normalized for overlay plot; if None, uses the original data.
    rolling_window: int - The window size for calculating the rolling median and percentiles (default is 10).
    cmap: str - Name of the colormap for target colors.
    '''

    run_numbers = range(1, len(data) + 1)

    if data_normalised is None:
        data_normalised = data

    # Load colormap for unique colors per target
    cmap = load_cmap(kwargs.get('cmap', default_qualitative_cmap))
    colors = random.sample(cmap.colors, len(targets))

    # Overlay plot with Min-Max scaled targets
    plt.figure(figsize=kwargs.get('fig_size', (6, 6)), tight_layout=True)
    for target in targets:
        plt.plot(run_numbers, data_normalised[target], color=colors[targets.index(target)], linewidth=0.5, linestyle=':',
                 marker='o', label=target)

    plt.xlabel('Run Number')
    plt.ylabel('Normalized Target Value')
    # plt.title('Targets vs. Run Number')
    plt.legend()
    if plot_saving_path is not None:
        plt.savefig(f'{plot_saving_path}/scatter_{target}_vs_experiment_number_V3.png', dpi=1200)
        plt.savefig(f'{plot_saving_path}/TimApproved{target}vsexp_number_V3.svg', format='svg', dpi=1200)
        plt.savefig(f'{plot_saving_path}/targets_vs_run_number_scaled.png')
    plt.show()

    # Separate plots_data1 for each target with shaded rolling median and percentiles
    for target in targets:
        values = data[target].dropna().values
        indices = np.arange(1, len(values) + 1)

        # Calculate the rolling median and percentile ranges
        rolling_median = pd.Series(values).rolling(window=rolling_window, min_periods=1).median()
        rolling_mean = pd.Series(values).rolling(window=rolling_window, min_periods=1).mean()
        lower_25 = pd.Series(values).rolling(window=rolling_window, min_periods=1).quantile(0.25)
        upper_75 = pd.Series(values).rolling(window=rolling_window, min_periods=1).quantile(0.75)
        lower_10 = pd.Series(values).rolling(window=rolling_window, min_periods=1).quantile(0.10)
        upper_90 = pd.Series(values).rolling(window=rolling_window, min_periods=1).quantile(0.90)

        plt.figure(figsize=kwargs.get('fig_size', (6, 6)), tight_layout=True)
        plt.scatter(indices, values, label=target, marker='o', color=colors[targets.index(target)])

        # Add shaded areas for percentile ranges
        plt.fill_between(indices, lower_25, upper_75, color=colors[targets.index(target)], alpha=0.2, label="25-75% Range")
        plt.fill_between(indices, lower_10, upper_90, color=colors[targets.index(target)], alpha=0.1, label="10-90% Range")

        # Plot the rolling median line
        plt.plot(indices, rolling_median, color=colors[targets.index(target)], linestyle="--", label=f"Rolling Median \n ({rolling_window} tests)")
        plt.plot(indices, rolling_mean, color=colors[targets.index(target)], linestyle=":", label=f"Rolling Mean \n ({rolling_window} tests)")

        plt.xlabel('Run Number')
        plt.ylabel(f'{target}')
        # plt.title(f'{target} vs. Run Number with Rolling Median and Percentile Bands')
        plt.legend()

        if plot_saving_path is not None:
            plt.savefig(f'{plot_saving_path}/{target}_vs_run_number.png')
        plt.show()

def plot_pareto_fronts(data: pd.DataFrame, targets: List[str], plot_saving_path: str = None,**kwargs) -> None:
    '''
    Plots Pareto fronts for given target combinations, displaying 2D or 3D Pareto fronts depending on the number of targets.

    Parameters:
    data: pd.DataFrame - Buchwald Mapping containing the target values to plot.
    targets: List[str] - List of target column names for which to plot the Pareto fronts.
    plot_saving_path: str, optional - Directory path to save the plots_data1 (if None, plots_data1 will not be saved).
    fig_size: Tuple[int, int], optional - Custom figure size for each plot, default is (8, 6).

    The function iterates through combinations of targets (up to 3 dimensions), plots_data1 the Pareto front for each
    combination, and optionally saves each plot to the specified path.
    '''

    fig_size = kwargs.get('fig_size', (6, 6))
    cmap_name = kwargs.get('cmap', default_sequential_cmap)
    cmap = load_cmap(cmap_name, cmap_type='continuous')  # Check if cmap is valid

    # Override colors if provided

    # check if any target is being minimised:
    to_minimise = kwargs.get('to_minimise', [])

    if len(targets) == 2:
        comb = tuple(targets)  # Only one combination, (target1, target2)
        plt.figure(figsize=fig_size, tight_layout=True)

        # Prepare data for plotting, adjusting targets that are to be minimized
        pareto_points = data[list(comb)].copy()
        for target in comb:
            if target in to_minimise:
                pareto_points[target] *= -1  # Flip values for targets to be minimized

        # 2D Pareto front
        scatter = plt.scatter(pareto_points[comb[0]], pareto_points[comb[1]], label='Buchwald Mapping Points',
                              c=range(len(data)), cmap=cmap, edgecolor='k')
        hull = ConvexHull(pareto_points)
        for simplex in hull.simplices:
            plt.plot(pareto_points.iloc[simplex, 0], pareto_points.iloc[simplex, 1], 'k:', linewidth=0.5)
        plt.xlabel(comb[0])
        plt.ylabel(comb[1])
        # plt.title(f'{comb[0]} vs. {comb[1]} with Pareto Front')
        plt.colorbar(scatter, label="Run Index")   # Add colorbar for 2D plot
        if plot_saving_path is not None:
            plt.savefig(f'{plot_saving_path}/{comb[0]}_vs_{comb[1]}_pareto_front.png')
        plt.show()

    else:
        # Generate both 2D and 3D plots_data1 for combinations of targets
        for comb in combinations(targets, 2):
            plt.figure(figsize=fig_size, tight_layout=True)

            # Prepare data for plotting, adjusting targets that are to be minimized
            pareto_points = data[list(comb)].copy()
            for target in comb:
                if target in to_minimise:
                    pareto_points[target] *= -1  # Flip values for targets to be minimized

            # 2D Pareto front
            scatter = plt.scatter(pareto_points[comb[0]], pareto_points[comb[1]], label='Buchwald Mapping Points',
                                  c=range(len(data)), cmap=cmap, edgecolor='k')
            hull = ConvexHull(pareto_points)
            for simplex in hull.simplices:
                plt.plot(pareto_points.iloc[simplex, 0], pareto_points.iloc[simplex, 1], 'k:', linewidth=0.5)
            plt.xlabel(comb[0])
            plt.ylabel(comb[1])
            # plt.title(f'{comb[0]} vs. {comb[1]} with Pareto Front')
            plt.colorbar(scatter, label="Run Index")   # Add colorbar for 2D plot
            if plot_saving_path is not None:
                plt.savefig(f'{plot_saving_path}/{comb[0]}_vs_{comb[1]}_pareto_front.png')
            plt.show()

        # Now generate 3D Pareto front plots_data1 for combinations of 3 targets
        for comb in combinations(targets, 3):
            fig = plt.figure(figsize=fig_size, constrained_layout=True)
            ax = fig.add_subplot(111, projection='3d')

            # Prepare data for plotting, adjusting targets that are to be minimized
            pareto_points = data[list(comb)].copy()
            for target in comb:
                if target in to_minimise:
                    pareto_points[target] *= -1  # Flip values for targets to be minimized

            # 3D Pareto front
            sc = ax.scatter(pareto_points[comb[0]], pareto_points[comb[1]], pareto_points[comb[2]],
                            label='Buchwald Mapping Points', c=range(len(data)), cmap=cmap, edgecolor='k')
            hull = ConvexHull(pareto_points)
            for simplex in hull.simplices:
                ax.plot(pareto_points.iloc[simplex, 0], pareto_points.iloc[simplex, 1], pareto_points.iloc[simplex, 2], 'k-')
            ax.set_xlabel(comb[0])
            ax.set_ylabel(comb[1])
            ax.set_zlabel(comb[2])
            # plt.title(f'{comb[0]} vs. {comb[1]} vs. {comb[2]} with Pareto Front')
            fig.colorbar(sc, ax=ax, label="Run Index")  # Add co
            if plot_saving_path is not None:
                plt.savefig(f'{plot_saving_path}/{comb[0]}_vs_{comb[1]}_vs_{comb[2]}_pareto_front.png')
            plt.show()

def plot_hypervolume_over_run_number(data: pd.DataFrame, targets: List[str], plot_saving_path: str = None, **kwargs) -> None:
    '''
    Plots the hypervolume over run number, based on convex hull volume for multi-target optimization.

    Parameters:
    data: pd.DataFrame - Buchwald Mapping containing the target values to plot.
    targets: List[str] - List of target column names for which to compute hypervolume.
    plot_saving_path: str, optional - Directory path to save the plot (if None, plot will not be saved).
    reference_point: np.ndarray, optional - Reference point for hypervolume calculation, default is slightly beyond [1.0, 1.0, ...].

    For each run number, the function calculates the hypervolume of the convex hull formed by targets.
    If there are insufficient points for a convex hull, it appends a 0 to maintain plot consistency.
    '''

    if len(targets) > 1:
        hypervolumes = []
        reference_point = kwargs.get('reference_point', np.array([1.1] * len(targets)))  # Slightly beyond normalized range

        for i in range(1, len(data) + 1):
            points = data[targets].iloc[:i].to_numpy()

            # Check if we have enough points to form a convex hull
            if points.shape[0] >= len(targets) + 1:
                hull = ConvexHull(points)
                hypervolumes.append(hull.volume)
            else:
                hypervolumes.append(0)  # Append 0 for early runs with insufficient points

        run_numbers = range(1, len(data) + 1)
        color = load_cmap(kwargs.get('cmap', default_qualitative_cmap)).colors[0]
        plt.figure(figsize=kwargs.get('fig_size', (6, 6)), tight_layout=True)
        plt.plot(run_numbers, hypervolumes, label='Hypervolume', color=color, marker='+', linestyle=':', linewidth=0.5)
        plt.xlabel('Run Number')
        plt.ylabel('Hypervolume')
        # plt.title('Hypervolume over Run Number')
        plt.legend()

        if plot_saving_path is not None:
            plt.savefig(f'{plot_saving_path}/hypervolume_vs_run_number.png')

        plt.show()

def plot_target_vs_number_tim_style(
    data: pd.DataFrame,
    targets: List[str],
    exploration_limit: int,
    old_data_limit: Optional[int] = None,
    plot_saving_path: Optional[str] = None,
    **kwargs
):
    '''
    Plots the data with a scatter plot for each target column (one per plot) against the index as x.
    Annotates the center of each dot with the index number and colorizes differently based on
    whether indexes are greater than or less than the exploration_limit.

    Parameters:
    - data: pd.DataFrame containing the data.
    - targets: List of target column names to plot.
    - exploration_limit: Integer index that separates the coloring of points.
    - plot_saving_path: Optional; path to save the plots_data1. If None, plots_data1 will be displayed.
    '''
    for target in targets:
        # Create a new figure
        plt.figure(figsize=kwargs.get("fig_size", (10, 6)))
        s = kwargs.get("s", 150)
        # x-axis: indices of the data
        indices = data.index.values+1

        # y-axis: values of the target column
        y_values = data[target].values
        to_minimise = kwargs.get("to_minimise", [])
        if target in to_minimise:
            y_values = y_values*-1
        y_values_var = None
        if (f"{target} Variance") in data.columns:
            y_values_var = data[f"{target} Variance"].values
            y_values_var = (y_values_var**0.5)/2

        # Colorize points based on exploration_limit
        colors_low = '#22265A'
        colors_high= '#F39103'
        colors_side= '#2AB34B'
        colors_old = '#02AA74'
        run_numbers = data.index.values + 1
    # limits & optional ranges from kwargs
        old_limit   = old_data_limit or 0
        init_end    = old_limit + exploration_limit
        side_limit  = kwargs.get("sideproduct_limit", None)
        second_rng  = kwargs.get("second_init_range", None)
        # second_rng = (start_idx, end_idx), both inclusive, 1-based.

        # ---- build masks ----
        old_mask   = run_numbers <= old_limit
        init_mask  = (run_numbers > old_limit) & (run_numbers <= init_end)

        # Second‐init block
        if second_rng:
            s0, s1        = second_rng
            second_mask   = (run_numbers >= s0) & (run_numbers <= s1)
        else:
            second_mask   = np.zeros_like(run_numbers, dtype=bool)

        # Sideproduct block
        if side_limit:
            side_mask     = run_numbers >= side_limit
        else:
            side_mask     = np.zeros_like(run_numbers, dtype=bool)

        # Optimisation splits:
        if second_rng:
            # Before second init:
            opt1_mask = (run_numbers > init_end) & (run_numbers < s0)
            # After second init (but before sideproduct):
            if side_limit:
                opt2_mask = (run_numbers > s1) & (run_numbers < side_limit)
            else:
                opt2_mask = run_numbers > s1
        else:
            # Single optimisation block if no second init
            opt1_mask = (run_numbers > init_end) & (
                (run_numbers < side_limit) if side_limit else True
            )
            opt2_mask = np.zeros_like(run_numbers, dtype=bool)

        # ==== replace your single plt.scatter(...) block with SIX scatter calls: ====
        if y_values_var is not None:
            plt.errorbar(indices[init_mask], y_values[init_mask], yerr=y_values_var[init_mask], fmt='none', ecolor=colors_low, capsize=1, elinewidth=2)
            plt.errorbar(indices[opt1_mask], y_values[opt1_mask], yerr=y_values_var[opt1_mask], fmt='none', ecolor=colors_high, capsize=1, elinewidth=2)
        # 1) Old data
        if old_mask.any():
            plt.scatter(indices[old_mask], y_values[old_mask],
                        s=s, c=colors_old, edgecolors='k', label='Old Buchwald Mapping')

        # 2) Initialization
        plt.scatter(indices[init_mask], y_values[init_mask],
                    s=s, c=colors_low, edgecolors='k', label='Initialization')

        # 3) Optimization 1
        plt.scatter(indices[opt1_mask], y_values[opt1_mask],
                    s=s, c=colors_high, edgecolors='k', label='Optimization ')

        # 4) Second Init (if present)
        if second_mask.any():
            plt.scatter(indices[second_mask], y_values[second_mask],
                        s=s, c=colors_low, edgecolors='k',
                        marker='^', label='Initialization B')

        # 5) Optimization 2 (if present)
        if opt2_mask.any():
            plt.scatter(indices[opt2_mask], y_values[opt2_mask],
                        s=s, c=colors_side, edgecolors='k', marker='^',
                        label='Optimization B')

        # 6) Sideproduct (if present)
        if side_mask.any():
            plt.scatter(indices[side_mask], y_values[side_mask],
                        s=s, c=colors_side, edgecolors='k',
                        marker='^', label='Optimization 9b')


        # plt.axvline(15.5, color = 'r', linestyle = '--')
        # plt.axvline(28.5, color = 'r', linestyle = '--')
        # plt.annotate('Exploit', ((15.5+28.5)/2, y_values.max()*1.1), fontsize = 15, color = 'k', ha = 'center')
        # plt.annotate('Explore', (30, y_values.max()*1.1), fontsize = 15, color = 'k', ha = "left")

        # Annotate each point with its index number
        # for idx, y in zip(indices, y_values):
        #     plt.annotate(
        #         str(idx),
        #         (idx, y),
        #         textcoords="offset points",
        #         xytext=(0, 0),
        #         ha='center',
        #         va='center',
        #         fontsize=0,
        #         color='white',
        #         weight='bold'
        #     )

        # Labels and title
        # plt.vlines([old_data_limit+exploration_limit+0.5], y_values.min() - 0.05*y_values.max(), y_values.max()*1.08, color = 'r', linestyle = '--')
        # plt.annotate("Recycled Buchwald Mapping from Screening", (old_data_limit/2, y_values.max()*1.05), fontsize = 10, color = 'r', ha = 'center')
        # plt.annotate("LHS", (old_data_limit + exploration_limit/2, y_values.max()*1.05), fontsize = 10, color = 'r', ha = 'center')
        # plt.annotate("Bayesian Optimisation", (old_data_limit + exploration_limit + (len(data) - old_data_limit - exploration_limit)/1.6, y_values.max()*1.05), fontsize = 10, color = 'r', ha = 'center')
        plt.xlabel('Experiment Number [#]')

        plt.ylabel(target)
        # plt.xticks(indices, indices)
        # plt.axvline(x= 31.5, linestyle='--', color='k', alpha = 0.7)
        # plt.title(f'Scatter Plot of {target} vs Experiment Number')
        min_ylim = 0 if target in ["Yield [%]", "Conversion [%]"] else min(y_values)
        max_ylim = 100 if target in ['Yield [%]', "Conversion [%]"] else max(y_values)*1.1

        plt.ylim(min_ylim, max_ylim)
        plt.tight_layout()
        plt.legend(loc = 'upper left')

        # Save or show the plot
        if plot_saving_path is not None:
                plt.savefig(f'{plot_saving_path}/scatter_{target}_vs_experiment_number_V3.png', dpi=1200)
                plt.savefig(f'{plot_saving_path}/TimApproved{target}vsexp_number_V3.svg', format='svg', dpi=1200)

        plt.show()

def plot_hypervolume_vs_run_number_tim_style(
    data: pd.DataFrame,
    targets: List[str],
    exploration_limit: int,
    plot_saving_path: Optional[str] = None,
    **kwargs
):
    '''
    Plots the hypervolume of the convex hull of the data points against the index as x.
    Annotates the center of each dot with the index number and colorizes differently based on
    whether indexes are greater than or less than the exploration_limit.

    Parameters:
    - data: pd.DataFrame containing the data.
    - targets: List of target column names to plot.
    - exploration_limit: Integer index that separates the coloring of points.
    - plot_saving_path: Optional; path to save the plots_data1. If None, plots_data1 will be displayed.
    '''
    hypervolumes = []
    s = kwargs.get('s', 40)
    sideprod_limit = kwargs.get('sideproduct_limit', None)
    reference_point = np.array([1.1] * len(targets))  # Slightly beyond normalized range

    for i in range(1, len(data) + 1):
        points = data[targets].iloc[:i].to_numpy()

        # Check if we have enough points to form a convex hull
        if points.shape[0] >= len(targets) + 1:
            hull = ConvexHull(points)
            hypervolumes.append(hull.volume)
        else:
            hypervolumes.append(0)  # Append 0 for early runs with insufficient points

    # Create a new figure
    plt.figure(figsize=kwargs.pop("fig_size", (9/2.54, 9/2.54)))

    # x-axis: indices of the data
    indices = data.index.values+1

    # Colorize points based on exploration_limit
    colors_low = '#22265A'
    colors_high= '#F39103'
    colors_side= '#2AB34B'

    # Create scatter plot
    plt.scatter(indices[:exploration_limit], hypervolumes[:exploration_limit], s = s, c = colors_low, edgecolors = 'k', label = 'Initialisation')
    if sideprod_limit is not None:
        plt.scatter(indices[exploration_limit:sideprod_limit], hypervolumes[exploration_limit:sideprod_limit], s = s, c = colors_high, edgecolors = 'k', label = 'Optimisation 13a')
        plt.scatter(indices[sideprod_limit:], hypervolumes[sideprod_limit:], s = s, c = colors_side,marker='^', edgecolors = 'k', label = 'Optimisation 13b')
    else:
        plt.scatter(indices[exploration_limit:], hypervolumes[exploration_limit:], s = s, c = colors_high, edgecolors = 'k', label = 'Optimisation')

    # # Annotate each point with its index number
    # for idx, y in zip(indices, hypervolumes):
    #     plt.annotate(
    #         str(idx),
    #         (idx, y),
    #         textcoords="offset points",
    #         xytext=(0, 0),
    #         ha='center',
    #         va='center',
    #         fontsize=8
    #     )

    # Labels and title
    plt.xlabel('Experiment Number [#]')
    plt.ylabel('Hypervolume [-]')
    plt.ylim(0, max(hypervolumes)*1.1)
    # plt.title(kwargs.pop("title",'Hypervolume over Experiment Number'))

    if plot_saving_path is not None:
        plt.savefig(f'{plot_saving_path}/hypervolume_vs_experiment_number_V3.png', dpi=1200)
        # plt.savefig(f'{plot_saving_path}/hypervolumevsexp_number_V3.svg', format='svg', dpi=1200)

    plt.show()


def plot_pareto_line(
        normalized_data: pd.DataFrame,
        not_normalized_data: pd.DataFrame,
        targets: List[str],
        plot_saving_path: Optional[str] = None,
        **kwargs
):
    """for the normalised data plots_data1 a line of target/target over the run number"""
    # find the cartesian product of the targets
    target_combinations = list(combinations(targets, 2))
    x_axis = normalized_data.index.values + 1

    for target_combination in target_combinations:
        target1, target2 = target_combination
        plt.figure(figsize=kwargs.pop("fig_size", (10, 6)))
        plt.plot(x_axis, normalized_data[target1] / normalized_data[target2], label=f'{target1}*{target2}', color='b', marker='o')
        plt.xlabel('Run Number')
        plt.ylabel(f'{target1}*{target2}')
        # plt.title(f'{target1}*{target2} vs. Run Number')

        if plot_saving_path is not None:
            plt.savefig(f'{plot_saving_path}/{target1}_over_{target2}_vs_run_number.png')
        # plt.ylim(0, 1E10)
        plt.axhline(1, color='r', linestyle='--', label = "Optimal")
        # plt.plot(x_axis, not_normalized_data[target1], color='r', linestyle='--', label = "Optimal")
        plt.legend()
        plt.show()


def plot_pareto_fronts_tim_style(
    data: pd.DataFrame,
    targets: List[str],
    exploration_limit: int,
    plot_saving_path: Optional[str] = None,
    **kwargs
):
    '''
    Plots Pareto fronts for the given 2D target combination.
    The data points are split into three groups:
      1. Optimisation Group 1 (Target 1 focused): plotted as circles.
      2. Optimisation Group 2 (Target 2 focused): plotted as triangles.
      3. Initialisation: plotted with a uniform color.

    The optimisation group is determined by `exploration_limit` (points before this index)
    and further split using the parameter `optim_split_index` (default: exploration_limit//2).
    Points with indices 0 to optim_split_index are target 1 focused (circles),
    and points from optim_split_index to exploration_limit are target 2 focused (triangles).

    Parameters:
    - data: pd.DataFrame containing the target values.
    - targets: List[str] of target column names (should be of length 2) for the Pareto plot.
    - exploration_limit: int - Index separating optimisation (indices < exploration_limit) from initialisation (indices >= exploration_limit).
    - plot_saving_path: Optional[str] - Directory where the plot should be saved. If None, the plot is only displayed.
    - kwargs: Additional keyword arguments, including:
        - fig_size: Tuple[int, int] for the figure size (default (6,6)).
        - s: marker size (default 50).
        - cmap: color map name (not used in this version, kept for compatibility).
        - to_minimise: list of target names where lower values are preferred.
        - optim_split_index: int index at which optimisation shifts from target 1 focused to target 2 focused.

    The plot shows the Pareto front using a convex hull and overlays the three groups with different markers and colors.
    '''

    fig_size = kwargs.get('fig_size', (6, 6))
    s = kwargs.get('s', 50)
    # Use two colors: one for initialisation and one for both optimisation groups.
    initialisation_color = '#22265A'
    optimisation_color = '#F39103'
    side_color = '#2AB34B'

    to_minimise = kwargs.get('to_minimise', [])
    # Determine the index at which optimisation focus shifts.
    # If not provided, default to half of the exploration_limit.
    optim_split_index = kwargs.get('optim_split_index', exploration_limit // 2)

    # Prepare the indices (1-based indexing)
    indices = data.index.values + 1


    # Work with a copy of the data to avoid modifying the original DataFrame.
    comb = tuple(targets)  # Expecting exactly two targets for a 2D plot.
    pareto_points = data[list(comb)].copy()

    # Adjust points for targets to be minimised (flip sign)
    for target in comb:
        if target in to_minimise:
            pareto_points[target] *= -1

    # Compute convex hull over all points (for the Pareto front line)

    plt.figure(figsize=fig_size, tight_layout=True)

    try:
        hull = ConvexHull(pareto_points)
        # Draw lines between the hull vertices
        for simplex in hull.simplices:
            plt.plot(pareto_points.iloc[simplex, 0], pareto_points.iloc[simplex, 1], 'k:', linewidth=0.5)
    except Exception as e:
        print("ConvexHull could not be computed:", e)



    # Split the data for optimisation and initialisation.
    # Here we assume that indices [0, exploration_limit) correspond to optimisation points,
    # and indices [exploration_limit, end) to initialisation.
    opt1 = pareto_points.iloc[exploration_limit:optim_split_index]
    opt2 = pareto_points.iloc[optim_split_index:]
    init_data = pareto_points.iloc[:exploration_limit]

    # Further split optimisation data based on the provided optim_split_index.
    # opt1 = optim_data.iloc[:optim_split_index]
    # opt2 = optim_data.iloc[optim_split_index:]

    # Plot the optimisation groups

    scatter_init = plt.scatter(init_data[comb[0]], init_data[comb[1]],
                               label='Initialisation',
                               c=initialisation_color, s=s, edgecolors='k')
    scatter_opt1 = plt.scatter(opt1[comb[0]], opt1[comb[1]],
                               label='Optimisation ',
                               c=optimisation_color, s=s, edgecolors='k', marker='o')

    # scatter_opt2 = plt.scatter(opt2[comb[0]], opt2[comb[1]],
    #                            label='Optimisation 9b',
    #                            c=side_color, s=s, edgecolors='k', marker='^')
    # # #
    # Plot the initialisation group (using the same points as before)


    # Optionally annotate each point with its experiment index (commented out by default)
    # for idx, x, y in zip(indices, pareto_points[comb[0]], pareto_points[comb[1]]):
    #     plt.annotate(
    #         str(idx),
    #         (x, y),
    #         textcoords="offset points",
    #         xytext=(0, 0),
    #         ha='center',
    #         va='center',
    #         fontsize=6,
    #         color='white',
    #         weight='bold'
    #     )


    plt.xlabel(f'{targets[0]}')
    plt.ylabel(f'{targets[1]}')
    plt.legend()

    if plot_saving_path is not None:
        # Save the plot as SVG in the specified directory
        plt.savefig(f'{plot_saving_path}/{comb[0]}_vs_{comb[1]}_pareto_front.png', dpi = 1200)

    plt.show()



def plot_targets_in_same_plot(
    data: pd.DataFrame,
    targets: List[str],
    exploration_limit: int,
    old_data_limit: Optional[int] = None,
    plot_saving_path: Optional[str] = None,
    **kwargs
):
    '''
    Plots all target columns on the same scatter plot against the index (experiment number).
    The y-values for each target are normalized to the range [0,1] independently.
    The data are colored according to the experiment grouping defined by old_data_limit and exploration_limit.
    Additionally, the second target is plotted using triangle markers while the others use circle markers.

    Parameters:
    - data: pd.DataFrame containing the data.
    - targets: List of target column names to plot.
    - exploration_limit: Integer number of experiments for the “initialisation” segment.
    - old_data_limit: Optional integer indicating how many points from the beginning belong to the “screening” segment.
                      If None, old_data_limit defaults to 0.
    - plot_saving_path: Optional; path to save the plot. If None, the plot will be displayed.
    - kwargs: Additional keyword arguments, such as:
        - fig_size: tuple for figure size (default (10, 6)).
        - s: marker size (default 150).
        - to_minimise: list of targets where the values should be multiplied by -1.

    The coloring for the experiment segments is:
        - Screening: color '#02AA74'
        - Initialisation: color '#22265A'
        - Optimisation: color '#F39103'
    '''

    # Set default for old_data_limit if not provided
    if old_data_limit is None:
        old_data_limit = 0

    # Define colors for each experiment segment
    colors_old = '#02AA74'       # Screening
    colors_low = '#22265A'       # Initialisation
    colors_high = '#F39103'      # Optimisation

    # Get additional plotting parameters
    fig_size = kwargs.get("fig_size", (10, 6))
    s = kwargs.get("s", 150)
    to_minimise = kwargs.get("to_minimise", [])

    # Prepare experiment indices (using 1-based indexing)
    indices = np.array(data.index.values) + 1

    # Create one figure for all targets
    plt.figure(figsize=fig_size)

    # To avoid duplicate legend entries for the experiment segments, track if label was already added.
    labels_added = { 'screening': False, 'initialisation': False, 'optimisation': False }

    # Iterate over each target and plot its segments
    for idx, target in enumerate(targets):
        # Retrieve the target values from the data
        y_values = data[target].values.astype(float)/100

        # If the target is to be minimized, multiply by -1 before normalization
        if target in to_minimise:
            y_values = -1 * y_values

        # Normalize the target values to [0,1] using the min and max of that target
        y_min, y_max = np.min(y_values), np.max(y_values)
        if y_max - y_min != 0:
            y_values_norm = (y_values - y_min) / (y_max - y_min)
        else:
            y_values_norm = np.full_like(y_values, 0.5)

        # Process variance if available: assume variance column named as "<target> variance"
        y_err_norm = None
        var_col = f"{target} variance"
        if var_col in data.columns:
            # Compute error as half the square root of the variance
            y_err = (np.sqrt(data[var_col].values) / 2)
            # Adjust error bars to normalized scale using the same range as the target
            if y_max - y_min != 0:
                y_err_norm = y_err / (y_max - y_min)
            else:
                y_err_norm = np.zeros_like(y_err)

        # Decide marker style: use triangle for the second target, circles for the others.
        marker_style = '^' if idx == 1 else 'o'

        # Determine index ranges for the three experiment segments
        screening_idx = indices[:old_data_limit]
        init_idx = indices[old_data_limit:old_data_limit + exploration_limit]
        opt_idx = indices[old_data_limit + exploration_limit:]

        screening_y = y_values_norm[:old_data_limit]
        init_y = y_values_norm[old_data_limit:old_data_limit + exploration_limit]
        opt_y = y_values_norm[old_data_limit + exploration_limit:]

        # Similarly for error bars if present
        if y_err_norm is not None:
            screening_err = y_err_norm[:old_data_limit]
            init_err = y_err_norm[old_data_limit:old_data_limit + exploration_limit]
            opt_err = y_err_norm[old_data_limit + exploration_limit:]

        # Plot each segment separately
        # Screening segment
        # label = f"{target} (Screening)" if not labels_added['screening'] else None
        # plt.scatter(screening_idx, screening_y, s=s, c=colors_old, edgecolors='k',
        #             marker=marker_style, label=label)
        # if y_err_norm is not None:
        #     plt.errorbar(screening_idx, screening_y, yerr=screening_err, fmt='none', ecolor=colors_old,
        #                  capsize=1, elinewidth=2)
        # labels_added['screening'] = True  # Ensure only one label per segment
        #
        # Initialisation segment
        label = f"{target} (Initialisation)" if not labels_added['initialisation'] else None
        plt.scatter(init_idx, init_y, s=s, c=colors_low, edgecolors='k',
                    marker=marker_style, label=label)
        if y_err_norm is not None:
            plt.errorbar(init_idx, init_y, yerr=init_err, fmt='none', ecolor=colors_low,
                         capsize=1, elinewidth=2)
        labels_added['initialisation'] = True

        # Optimisation segment
        label = f"{target} (Optimisation)" if not labels_added['optimisation'] else None
        plt.scatter(opt_idx, opt_y, s=s, c=colors_high, edgecolors='k',
                    marker=marker_style, label=label)
        # plt.plot(opt_idx, opt_y, color = colors_high, linestyle = ':', linewidth = 0.5)
        if y_err_norm is not None:
            plt.errorbar(opt_idx, opt_y, yerr=opt_err, fmt='none', ecolor=colors_high,
                         capsize=1, elinewidth=2)
        labels_added['optimisation'] = True

    # Labels and ticks
    plt.xlabel('Experiment Number [#]')
    plt.ylabel('Yield [%]')
    # plt.xticks(indices, indices)
    plt.yticks(np.linspace(0, 1, 11), [f"{i/10:.1f}" for i in range(11)])
    plt.tight_layout()
    plt.ylim(0, 1.1)

    # Add legend
    # plt.legend()

    # Save or display the plot
    if plot_saving_path is not None:
        # Save the plot as PNG and SVG
        plt.savefig(f'{plot_saving_path}/combined_targets_scatter.png', dpi=1200)
        plt.savefig(f'{plot_saving_path}/combined_targets_scatter.svg', format='svg', dpi=1200)
    plt.show()

def plot_pareto_fronts_tim_style_with_colorbar(
    data: pd.DataFrame,
    targets: List[str],
    exploration_limit: int,
    plot_saving_path: Optional[str] = None,
    **kwargs
):
    '''
    Plots Pareto fronts for the given 2D target combination, with the third dimension shown as color.
    Initialisation: circles
    Optimisation: triangles
    '''

    assert len(targets) == 3, "Expected exactly 3 targets for 2D plot + colorbar."

    fig_size = kwargs.get('fig_size', (7, 6))
    s = kwargs.get('s', 50)
    initialisation_color = '#22265A'
    optimisation_color = '#F39103'
    to_minimise = kwargs.get('to_minimise', [])
    optim_split_index = kwargs.get('optim_split_index', exploration_limit // 2)

    comb = tuple(targets[:2])  # First two targets for 2D plot
    color_dim = targets[2]     # Third target for colorbar

    pareto_points = data[list(comb) + [color_dim]].copy()
    for target in comb:
        if target in to_minimise:
            pareto_points[target] *= -1

    fig, ax = plt.subplots(figsize=fig_size, tight_layout=True)

    try:
        hull = ConvexHull(pareto_points[comb])
        for simplex in hull.simplices:
            ax.plot(pareto_points.iloc[simplex, 0], pareto_points.iloc[simplex, 1], 'k:', linewidth=0.5)
    except Exception as e:
        print("ConvexHull could not be computed:", e)

    # Split
    optim_data = pareto_points.iloc[:exploration_limit]
    init_data = pareto_points.iloc[exploration_limit:]

    opt1 = optim_data.iloc[:optim_split_index]
    opt2 = optim_data.iloc[optim_split_index:]

    # Plot initialisation
    scatter_init = ax.scatter(
        init_data[comb[0]], init_data[comb[1]],
        c=init_data[color_dim], cmap='magma', s=s,
        marker='o', edgecolors='k', label='Initialisation'
    )

    # Plot optimisation groups
    scatter_opt1 = ax.scatter(
        opt1[comb[0]], opt1[comb[1]],
        c=opt1[color_dim], cmap='magma', s=s,
        marker='^', edgecolors='k', label='Optimisation'
    )

    # scatter_opt2 = ax.scatter(
    #     opt2[comb[0]], opt2[comb[1]],
    #     c=opt2[color_dim], cmap='viridis', s=s,
    #     marker='^', edgecolors='k', label='Optimisation'
    # )

    ax.set_xlabel(f'{comb[0]}')
    ax.set_ylabel(f'{comb[1]}')
    ax.legend()

    # Add colorbar
    norm = plt.Normalize(data[color_dim].min(), data[color_dim].max())
    sm = plt.cm.ScalarMappable(cmap='magma', norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label(color_dim)

    if plot_saving_path is not None:
        plt.savefig(f'{plot_saving_path}/{comb[0]}_vs_{comb[1]}_colored_by_{color_dim}_pareto.svg')

    plt.show()






def plot_pareto_tim_style_3d_projections(
    data: pd.DataFrame,
    targets: List[str],
    exploration_limit: int,
    plot_saving_path: Optional[str] = None,
    **kwargs
):
    '''
    Projects 2D Pareto fronts onto the 3 planes (XY, XZ, YZ) in a 3D plot, with Tim's style:
    - Initialisation = dark blue circles
    - Optimisation = orange circles
    - Convex hull = dotted lines
    '''
    assert len(targets) == 3, "Need exactly 3 targets for 3D projection."

    fig_size = kwargs.get('fig_size', (8, 6))
    s = kwargs.get('s', 50)
    to_minimise = kwargs.get('to_minimise', [])
    optim_split_index = kwargs.get('optim_split_index', exploration_limit // 2)

    initialisation_color = '#22265A'
    optimisation_color = '#F39103'
    # side_color = '#2AB34B'  # not used

    X, Y, Z = targets
    pareto_points = data[[X, Y, Z]].copy()

    for t in to_minimise:
        if t in pareto_points.columns:
            pareto_points[t] *= -1

    init_data = pareto_points.iloc[:exploration_limit]
    opt_data = pareto_points.iloc[exploration_limit:]

    opt1 = opt_data.iloc[:optim_split_index]
    opt2 = opt_data.iloc[optim_split_index:]

    fig = plt.figure(figsize=fig_size, tight_layout=True)
    ax = fig.add_subplot(111, projection='3d')

    def draw_2d_pareto(ax, df, a, b, fixed_dim, fixed_val, color_init, color_opt):
        proj_df = df[[a, b]].copy()
        proj_df['fixed'] = fixed_val

        # Convex hull
        try:
            hull = ConvexHull(proj_df[[a, b]])
            for simplex in hull.simplices:
                p1 = proj_df.iloc[simplex[0]]
                p2 = proj_df.iloc[simplex[1]]
                xs, ys = [p1[a], p2[a]], [p1[b], p2[b]]
                lineargs = {'linestyle': ':', 'color': 'k', 'linewidth': 1}
                if fixed_dim == 'x':
                    ax.plot([fixed_val]*2, xs, ys, **lineargs)
                elif fixed_dim == 'y':
                    ax.plot(xs, [fixed_val]*2, ys, **lineargs)
                else:  # 'z'
                    ax.plot(xs, ys, [fixed_val]*2, **lineargs)
        except Exception as e:
            print(f"Convex hull on {a} vs {b} failed:", e)

        # Initialisation
        init = proj_df.iloc[:exploration_limit]
        opt = proj_df.iloc[exploration_limit:]

        if fixed_dim == 'x':
            ax.scatter([fixed_val]*len(init), init[a], init[b], c=color_init, s=s, edgecolors='k', marker='o', label=f'Initialization')
            ax.scatter([fixed_val]*len(opt), opt[a], opt[b], c=color_opt, s=s, edgecolors='k', marker='o', label=f'Optimization')
        elif fixed_dim == 'y':
            ax.scatter(init[a], [fixed_val]*len(init), init[b], c=color_init, s=s, edgecolors='k', marker='o')
            ax.scatter(opt[a], [fixed_val]*len(opt), opt[b], c=color_opt, s=s, edgecolors='k', marker='o')
        else:  # 'z'
            ax.scatter(init[a], init[b], [fixed_val]*len(init), c=color_init, s=s, edgecolors='k', marker='o')
            ax.scatter(opt[a], opt[b], [fixed_val]*len(opt), c=color_opt, s=s, edgecolors='k', marker='o')

    # Project onto XY (Z=0), XZ (Y=0), YZ (X=0)
    draw_2d_pareto(ax, pareto_points, X, Y, 'z', 0, initialisation_color, optimisation_color)
    draw_2d_pareto(ax, pareto_points, X, Z, 'y', 0, initialisation_color, optimisation_color)
    draw_2d_pareto(ax, pareto_points, Y, Z, 'x', 0, initialisation_color, optimisation_color)

    ax.set_xlabel(X)
    ax.set_ylabel(Y)
    ax.set_zlabel(Z)
    ax.view_init(elev=30, azim=135)
    ax.legend(loc='upper left')
    ax.grid(False)
    ax.set_xlim(100,0)
    ax.set_ylim(0,5)
    ax.set_zlim(0, 100)


    if plot_saving_path:
        plt.savefig(f"{plot_saving_path}/{X}_{Y}_{Z}_pareto_tim_3d_projection.svg", dpi=1200, format='svg')

    plt.show()

def plot_pareto_2d_colored_by_third(
    data: pd.DataFrame,
    targets: List[str],
    exploration_limit: int,
    plot_saving_path: Optional[str] = None,
    **kwargs
):
    """
    Plots 2D Pareto front with color representing the third variable.
    Initialisation: squares
    Optimisation: circles
    """
    assert len(targets) == 3, "Expected exactly 3 targets (X, Y, color)."

    X, Y, C = targets
    fig_size = kwargs.get('fig_size', (6, 6))
    s = kwargs.get('s', 50)
    to_minimise = kwargs.get('to_minimise', [])

    # Flip direction for targets to minimise
    pareto_points = data[[X, Y, C]].copy()
    for t in to_minimise:
        if t in pareto_points.columns:
            pareto_points[t] *= -1

    # Split data
    init_data = pareto_points.iloc[:exploration_limit]
    opt_data = pareto_points.iloc[exploration_limit:]

    # Convex hull on X-Y
    try:
        hull = ConvexHull(pareto_points[[X, Y]])
        hull_pts = pareto_points[[X, Y]].iloc[hull.vertices]
    except Exception as e:
        print("Convex hull could not be computed:", e)
        hull_pts = None

    fig, ax = plt.subplots(figsize=fig_size, tight_layout=True)

    # Plot convex hull lines
    if hull_pts is not None:
        for simplex in hull.simplices:
            pts = pareto_points[[X, Y]].iloc[simplex]
            ax.plot(pts[X], pts[Y], 'k:', linewidth=0.5)

    # Plot initialisation (squares)
    scatter_init = ax.scatter(
        init_data[X], init_data[Y],
        c=init_data[C],
        cmap='magma_r',
        s=s,
        edgecolors='k',
        marker='s',
        label='Initialisation'
    )

    # Plot optimisation (circles)
    scatter_opt = ax.scatter(
        opt_data[X], opt_data[Y],
        c=opt_data[C],
        cmap='magma_r',
        s=s,
        edgecolors='k',
        marker='o',
        label='Optimisation'
    )

    cbar = plt.colorbar(scatter_opt, ax=ax)
    cbar.set_label(C)

    ax.set_xlabel(X)
    ax.set_ylabel(Y)
    ax.legend()

    if plot_saving_path:
        plt.savefig(f"{plot_saving_path}/{X}_vs_{Y}_colored_by_{C}_pareto.png", dpi=1200)

    plt.show()

# Normalize target columns
scaler = MinMaxScaler()
# data['Integral sideproduct'] = data['Integral sideproduct'] * -1
targets_transformed = pd.DataFrame(
    data=scaler.fit_transform(data[targets]),  # Fit and transform all targets at once
    columns=targets  # Preserve column names
)


plot_target_vs_number_tim_style(data = data, targets=targets, exploration_limit=exploration_limit, old_data_limit = None, plot_saving_path=plot_saving_path, fig_size=(10/2.54, 10/2.54) ,
                                s = 30,
                                sideproduct_limit=sideprod_start,
                                # second_init_range=(32,35)
)


# plot_targets_in_same_plot(
#     data = data,
#     targets = targets,
#     exploration_limit=exploration_limit,
#     old_data_limit=None,
#     plot_saving_path = plot_saving_path,
#     fig_size=(10/2.54,10/2.54),
#     s = 40
# )
# plot_targets_vs_run_number(data=data, targets=targets, data_normalised=targets_transformed, plot_saving_path=plot_saving_path)
# # plot_pareto_fronts(data=targets_transformed, targets=targets, to_minimise=minimise, plot_saving_path=plot_saving_path)
# plot_hypervolume_vs_run_number_tim_style(data=targets_transformed, targets=targets, exploration_limit=exploration_limit, plot_saving_path=plot_saving_path, s = 30,
#                                         fig_size=(10/2.54, 10/2.54),
#                                         sideproduct_limit=sideprod_start,
#                                          )
# # # # # # # # # # targets_inverted = targets[-1::-1]
# tar_couples = [(0,1), (0,2), (1,2)]
tar_couples = [(0,1)]
for tar in tar_couples:
    targets_run = [targets[tar[0]], targets[tar[1]]]
    plot_pareto_fronts_tim_style(data=data, targets=targets_run, exploration_limit=exploration_limit, plot_saving_path=plot_saving_path,
                             fig_size=(10/2.54, 10/2.54),
                             s = 40,
                             optim_split_index=500
                             )
#
# plot_pareto_fronts_tim_style_with_colorbar(
#     data = data,
#     targets=targets,
#     exploration_limit=exploration_limit,
#     plot_saving_path=plot_saving_path,
#     fig_size=(10/2.54, 10/2.54),
#     s=40,
#     optim_split_index=200
# )


# plot_pareto_tim_style_3d_projections(
#     data = data,
#     targets=targets,
#     exploration_limit=exploration_limit,
#     plot_saving_path=plot_saving_path,
#     fig_size=(5, 5),
#     s=40
# )
# plot_pareto_2d_colored_by_third(
#     data = data,
#     targets=[targets[0], targets[2], targets[1]],
#     exploration_limit=exploration_limit,
#     plot_saving_path=plot_saving_path,
#     fig_size=(5, 5),
#     s=40
# )
# plot_hypervolume_over_run_number(data=targets_transformed, targets=targets, plot_saving_path=plot_saving_path)
# plot_pareto_line(normalized_data=targets_transformed, not_normalized_data=data, targets=targets, plot_saving_path=plot_saving_path)
# print(data_copy.iloc[-1])

In [ ]:
def plot_pareto_front_3d_with_convex_hull(
    data: pd.DataFrame,
    targets: List[str],
    exploration_limit: int,
    plot_saving_path: Optional[str] = None,
    **kwargs
):
    '''
    Plots a 3D Pareto front using the three targets as X, Y, Z coordinates.
    Convex hull is drawn in 3D.
    Initialisation: circles
    Optimisation: triangles
    '''

    assert len(targets) == 3, "Expected exactly 3 targets for 3D Pareto plot."

    fig_size = kwargs.get('fig_size', (8, 6))
    s = kwargs.get('s', 50)
    to_minimise = kwargs.get('to_minimise', [])
    optim_split_index = kwargs.get('optim_split_index', exploration_limit // 2)

    pareto_points = data[targets].copy()
    for target in targets:
        if target in to_minimise:
            pareto_points[target] *= -1

    X, Y, Z = targets
    points = pareto_points[[X, Y, Z]].values

    fig = plt.figure(figsize=fig_size, tight_layout=True)
    ax = fig.add_subplot(111, projection='3d')

    try:
        hull = ConvexHull(points)
        for s in hull.simplices:
            tri = Poly3DCollection([points[s]], alpha=0.05, edgecolor='k')
            tri.set_facecolor('#bbbbbb')
            ax.add_collection3d(tri)
    except Exception as e:
        print("3D ConvexHull could not be computed:", e)

    # Split into init and optim
    init_data = pareto_points.iloc[:exploration_limit]
    optim_data = pareto_points.iloc[exploration_limit:]
    # opt1 = optim_data.iloc[:optim_split_index]
    # opt2 = optim_data.iloc[optim_split_index:]

    ax.scatter(init_data[X], init_data[Y], init_data[Z], c='#8b97cc', s=60, marker='o', edgecolor='k', label='Initialisation')
    ax.scatter(optim_data[X], optim_data[Y], optim_data[Z], c='#fadeb9', s=60, marker='o', edgecolor='k', label='Optimisation')
    # ax.scatter(opt2[X], opt2[Y], opt2[Z], c='#F39103', s=s, marker='^', edgecolor='k', label='Optimisation Sideproduct')
    def draw_projection_lines(ax, df, color='gray', alpha=0.8, lw=2):
        for _, row in df.iterrows():
            x, y, z = row[X], row[Y], row[Z]

            # # Drop to XY plane
            ax.plot([x, x], [y, y], [z, 0], color=color, alpha=alpha, lw=lw, linestyle=':')
            #
            # # Drop to XZ plane
            # ax.plot([x, x], [y, 0], [z, z], color=color, alpha=alpha, lw=lw)

            # Drop to YZ plane
            # ax.plot([x, 0], [y, y], [z, z], color=color, alpha=alpha, lw=lw)

    draw_projection_lines(ax, optim_data, color='#fadeb9')
    draw_projection_lines(ax, init_data, color='#8b97cc')
    ax.set_xlabel(X)
    ax.set_ylabel(Y)
    ax.set_zlabel(Z)
    ax.legend()
    ax.set_ylim(0,6)
    ax.set_xlim(100,0)
    ax.set_zlim(0, 100)
    ax.view_init(elev=20, azim=100)
    # Change grid color
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis._axinfo['grid']['color'] = '#000000'
        axis._axinfo['grid']['linestyle'] = ':'
        axis._axinfo['grid']['linewidth'] = 0.5

    try:
        ax.xaxis.pane.set_facecolor('white')
        ax.yaxis.pane.set_facecolor('white')
        ax.zaxis.pane.set_facecolor('white')

        ax.xaxis.pane.set_edgecolor('white')
        ax.yaxis.pane.set_edgecolor('white')
        ax.zaxis.pane.set_edgecolor('white')
    except Exception as e:
        print("Couldn't set 3D pane facecolors:", e)
    # Set background color



    if plot_saving_path is not None:
        plt.savefig(f'{plot_saving_path}/{X}_vs_{Y}_vs_{Z}_pareto_3d.svg')

    plt.show()

# plot_pareto_front_3d_with_convex_hull(
#     data = data,
#     targets=targets,
#     exploration_limit=exploration_limit,
#     plot_saving_path=plot_saving_path,
#     fig_size=(15/2.54, 15/2.54),
#     s=40,
#     optim_split_index=200
# )

# Plots of choiches
Now that we plotted the results, we should also be plotting the choiches that the algo made:
- histogram of value pickings per value
- choiches over run

In [ ]:
def plot_histogram_bar(data: pd.DataFrame, categorical_cols: List[str], continuous_cols: List[str], 
                       plot_saving_path:None | str = None, **kwargs) -> None:
    """
    Plots histograms for continuous columns and bar plots_data1 for categorical columns in a grid layout.
    
    Parameters:
    data: pd.DataFrame - The DataFrame containing the data to plot.
    categorical_cols: List[str] - List of column names for categorical variables.
    continuous_cols: List[str] - List of column names for continuous variables.
    plot_saving_path: Optional[str] - Path to save the plot image. If None, the plot is not saved.
    colormap: str - Name of the pypalettes colormap to use for colorizing plots_data1.
    
    The function arranges the plots_data1 in a grid with two columns. Continuous variables are displayed with
    histograms and their mean and mode values highlighted. Categorical variables are displayed as count plots_data1.
    Each plot is assigned a different color from the colormap.
    """
    valid_categorical_cols = [col for col in categorical_cols if data[col].nunique() > 1]
    valid_continuous_cols = [col for col in continuous_cols if data[col].nunique() > 1 and 'Variance' not in col]
    total_plots = len(valid_continuous_cols) + len(valid_categorical_cols)
    cols = 2  # Number of columns in the grid
    rows = math.ceil(total_plots / cols)
    
    fig_size = kwargs.get('fig_size', (12, 4))
    # Set up the figure with subplots
    fig, axes = plt.subplots(rows, cols, figsize=(fig_size[0], rows * fig_size[1]), tight_layout = True)
    axes = axes.flatten()  # Flatten the axes array for easy indexing
    
    # Generate colors from the colormap
    cmap = load_cmap(kwargs.get('colormap', default_sequential_cmap))
    colors = random.sample(cmap.colors, total_plots)
    colors = kwargs.get('colors', colors)

    for i, col in enumerate(valid_continuous_cols + valid_categorical_cols):
        color = colors[i]  # Get a unique color for each plot

        if col in continuous_cols and 'light' not in col.lower():
            sns.histplot(data[col], ax=axes[i], kde=True, color=color)
            axes[i].axvline(data[col].mean(), color='r', linestyle=':', label='Mean')
            axes[i].axvline(data[col].median(), color='g', linestyle='--', label='Median')
                        
            axes[i].legend()
        
        else:
            sns.countplot(x=col, data=data, ax=axes[i], color=color)
        axes[i].tick_params(axis='x', rotation=45)
        axes[i].set_title(f'Distribution of {col}')

    
    # Hide any unused axes if there are extra spaces in the grid
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    if plot_saving_path is not None:
        name_plus = kwargs.get('name_plus', '')
        plt.savefig(f'{plot_saving_path}/{name_plus}choices_histogram_bar.png', dpi=1200)
        # plt.savefig(f'{plot_saving_path}/{name_plus}choices_histogram_bar.svg', dpi=1200)
    plt.show()
    if kwargs.get('return_colors', False):
        return colors



def plot_statistical_funnel_vs_index(data: pd.DataFrame, continuous_cols: List[str], categorical_cols: List[str], 
                                     plot_saving_path: None|str = None, colormap: str = 'viridis', **kwargs) -> None:
    """
    Plots continuous columns as a statistical "funnel" over index, with a median line and percentile bands,
    and categorical columns as discrete choice points over index.

    Parameters:
    data: pd.DataFrame - The DataFrame containing the data to plot.
    continuous_cols: List[str] - List of column names with continuous values to plot as funnels.
    categorical_cols: List[str] - List of column names with categorical values to plot as discrete choices.
    plot_saving_path: Optional[str] - Path to save the plot image. If None, the plot is not saved.
    colormap: str - Name of the matplotlib colormap to use for colorizing plots_data1, default is 'viridis'.

    The function arranges the plots_data1 in a grid, displaying continuous values with a median line and percentile bands
    to create a funnel effect, and categorical values as discrete choice points.
    """

    valid_categorical_cols = [col for col in categorical_cols if data[col].nunique() > 1]
    valid_continuous_cols = [col for col in continuous_cols if data[col].nunique() > 1]
    total_plots = len(valid_continuous_cols) + len(valid_categorical_cols)

    cols = 2  # Number of columns in the grid layout
    rows = math.ceil(total_plots / cols)

    # Set up the figure with subplots
    fig, axes = plt.subplots(rows, cols, figsize=(12, rows * 4), constrained_layout=True)
    axes = axes.flatten()  # Flatten the axes array for easy indexing

    # Generate colors from the colormap
    cmap = plt.get_cmap(colormap)
    colors = [cmap(i / total_plots) for i in range(total_plots)]
    colors = kwargs.get('colors', colors)

    for i, col in enumerate(valid_continuous_cols + valid_categorical_cols):
        color = colors[i]  # Get a unique color for each plot

        if col in valid_continuous_cols:
            # Plot continuous data as a statistical funnel plot
            values = data[col].dropna().values
            indices = np.arange(len(values))

            # Calculate the median and percentiles for the funnel
            median_values = pd.Series(values).expanding().median()
            lower_25 = pd.Series(values).expanding().quantile(0.25)
            upper_75 = pd.Series(values).expanding().quantile(0.75)
            lower_10 = pd.Series(values).expanding().quantile(0.10)
            upper_90 = pd.Series(values).expanding().quantile(0.90)

            # Plot the main line and shaded percentile bands
            sns.lineplot(x=indices, y=values, ax=axes[i], color=color, marker="o", linestyle="", alpha=0.6)
            axes[i].plot(indices, median_values, color=color, linestyle="--", label="Median")
            axes[i].fill_between(indices, lower_25, upper_75, color=color, alpha=0.2, label="25-75% Range")
            axes[i].fill_between(indices, lower_10, upper_90, color=color, alpha=0.1, label="10-90% Range")

            axes[i].set_title(f'{col} Statistical Funnel Plot')
            axes[i].set_xlabel('Index')
            axes[i].set_ylabel(col)
            axes[i].legend()

        else:
            # Plot categorical data as discrete points on y-axis
            unique_categories = sorted(data[col].dropna().unique())
            category_mapping = {cat: idx for idx, cat in enumerate(unique_categories)}

            mapped_values = data[col].map(category_mapping)
            sns.scatterplot(x=data.index, y=mapped_values, ax=axes[i], color=color, marker="o")
            axes[i].set_yticks(list(category_mapping.values()))
            axes[i].set_yticklabels(list(category_mapping.keys()))
            axes[i].set_title(f'Discrete Choices of {col} Over Index')
            axes[i].set_xlabel('Index')
            axes[i].set_ylabel(col)

    # Hide any unused axes if there are extra spaces in the grid
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    if plot_saving_path is not None:
        plt.savefig(f'{plot_saving_path}/column_statistical_funnel_vs_index.png')
    plt.show()

def plot_statistical_funnel_dynamic_vs_index(data: pd.DataFrame, continuous_cols: List[str], categorical_cols: List[str], 
                                     plot_saving_path: None|str = None, n: int = 3, **kwargs) -> None:
    """
    Plots continuous columns as a statistical "funnel" over index with a rolling median over the last n tests,
    and categorical columns as discrete choice points over index.

    Parameters:
    data: pd.DataFrame - The DataFrame containing the data to plot.
    continuous_cols: List[str] - List of column names with continuous values to plot as funnels.
    categorical_cols: List[str] - List of column names with categorical values to plot as discrete choices.
    plot_saving_path: Optional[str] - Path to save the plot image. If None, the plot is not saved.
    colormap: str - Name of the matplotlib colormap to use for colorizing plots_data1, default is 'viridis'.
    n: int - Number of tests to consider for calculating the rolling median (default is 10).

    The function arranges the plots_data1 in a grid, displaying continuous values with a rolling median line and
    percentile bands to create a funnel effect, and categorical values as discrete choice points.
    """
    valid_categorical_cols = [col for col in categorical_cols if data[col].nunique() > 1]
    valid_continuous_cols = [col for col in continuous_cols if data[col].nunique() > 1 and 'Variance' not in col]
    total_plots = len(valid_continuous_cols) + len(valid_categorical_cols)

    cols = 2  # Number of columns in the grid layout
    rows = math.ceil(total_plots / cols)

    fig_size = kwargs.get('fig_size', (12, 4))
    # Set up the figure with subplots
    fig, axes = plt.subplots(rows, cols, figsize=(fig_size[0], rows * fig_size[1]))
    axes = axes.flatten()  # Flatten the axes array for easy indexing

    # Generate colors from the colormap
    cmap = load_cmap(kwargs.get('colormap', default_qualitative_cmap))
    colors = random.sample(cmap.colors, total_plots)

    colors = kwargs.get('colors', colors)

    for i, col in enumerate(valid_continuous_cols + valid_categorical_cols):
        color = colors[i]  # Get a unique color for each plot

        if col in valid_continuous_cols and 'light' not in col.lower():
            # Plot continuous data as a statistical funnel plot with a rolling median over the last n tests
            values = data[col].dropna().values
            indices = np.arange(len(values))

            # Calculate the rolling median and percentiles over the last n tests
            rolling_median = pd.Series(values).rolling(window=n, min_periods=1).median()
            rolling_mean = pd.Series(values).rolling(window=n, min_periods=1).mean()
            lower_25 = pd.Series(values).rolling(window=n, min_periods=1).quantile(0.25)
            upper_75 = pd.Series(values).rolling(window=n, min_periods=1).quantile(0.75)
            lower_10 = pd.Series(values).rolling(window=n, min_periods=1).quantile(0.10)
            upper_90 = pd.Series(values).rolling(window=n, min_periods=1).quantile(0.90)

            # Plot the main line and shaded percentile bands
            sns.lineplot(x=indices, y=values, ax=axes[i], color=color, marker="o", linestyle="", alpha=0.6)
            axes[i].plot(indices, rolling_median, color=color, linestyle="--", label=f"Rolling Median ({n} tests)")
            axes[i].plot(indices, rolling_mean, color=color, linestyle=":", label=f"Rolling Mean ({n} tests)")
            axes[i].fill_between(indices, lower_25, upper_75, color=color, alpha=0.2, label="25-75% Range")
            axes[i].fill_between(indices, lower_10, upper_90, color=color, alpha=0.1, label="10-90% Range")

            axes[i].set_title(f'{col}')
            axes[i].set_xlabel('Index')
            axes[i].set_ylabel(col)
            axes[i].legend()

        else:
            unique_categories = sorted(data[col].dropna().unique())
            category_mapping = {cat: idx for idx, cat in enumerate(unique_categories)}
            mapped_values = data[col].map(category_mapping)

            # Check if mapped_values has meaningful data
            if mapped_values.dropna().nunique() > 1:
                # Compute the rolling mode using a custom function if there is variability
                rolling_modes = [
                    mode(mapped_values[max(0, j - n + 1):j + 1].dropna()).mode 
                    if not mapped_values[max(0, j - n + 1):j + 1].dropna().empty else np.nan
                    for j in range(len(mapped_values))
                ]
                
                # Scatter plot for actual values and rolling mode
                sns.scatterplot(x=data.index, y=mapped_values, ax=axes[i], color=color, marker="o", label="Actual Choices")
                sns.lineplot(x=data.index, y=rolling_modes, ax=axes[i], color=color, linestyle="--", label=f"Rolling Mode (last {n} tests)")
            else:
                # Plot only the actual values if there’s only one unique value or no values
                sns.scatterplot(x=data.index, y=mapped_values, ax=axes[i], color=color, marker="o", label="Actual Choices")

            # Set the y-axis labels for categories
            axes[i].set_yticks(list(category_mapping.values()))
            axes[i].set_yticklabels(list(category_mapping.keys()), rotation=45)
            axes[i].set_title(f'Rolling Mode of {col} Choices Over Index')
            axes[i].set_xlabel('Index')
            axes[i].set_ylabel(col)
            axes[i].legend()
    # Hide any unused axes if there are extra spaces in the grid
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    if plot_saving_path is not None:
        name_plus = kwargs.get('name_plus', '')
        plt.savefig(f'{plot_saving_path}/{name_plus}column_statistical_funnel_vs_index.png', dpi=1200)
        # plt.savefig(f'{plot_saving_path}/{name_plus}column_statistical_funnel_vs_index.svg', dpi=1200)
    plt.show()
    if kwargs.get('return_colors', False):
        return colors

# break data up


colors = plot_histogram_bar(data, categorical_cols=colum_types['categorical'], continuous_cols=colum_types['numerical'], return_colors=True,plot_saving_path=plot_saving_path, colormap='blaziken')

# plot_histogram_bar(data_2, categorical_cols=colum_types['categorical'], continuous_cols=colum_types['numerical'], return_colors=True,plot_saving_path=plot_saving_path, colors = colors, name_plus = 'data_2')
# plot_column_funnel_vs_index(data, categorical_cols=colum_types['categorical'], continuous_cols=colum_types['numerical'])
# plot_column_concave_funnel_vs_index(data, categorical_cols=colum_types['categorical'], continuous_cols=colum_types['numerical'], alpha=0.01)
# plot_statistical_funnel_vs_index(data, categorical_cols=colum_types['categorical'], continuous_cols=colum_types['numerical'])
plot_statistical_funnel_dynamic_vs_index(data, categorical_cols=colum_types['categorical'], continuous_cols=colum_types['numerical'], n= 5, colors=colors, plot_saving_path=plot_saving_path,)
# raise ValueError()
# plot_statistical_funnel_dynamic_vs_index(data_2, categorical_cols=colum_types['categorical'], continuous_cols=colum_types['numerical'], n= 5, colors=colors, plot_saving_path=plot_saving_path, name_plus = 'data_2')

# Polar Plotting of Sensitivity Analysis

This set of plots, we are plotting all the possible discrete and continuous choiches and how much they make it vary from the optimum:


In [ ]:
def plot_deviation_vs_distance(
    data: pd.DataFrame,
    categorical_cols: list,
    continuous_cols: list,
    targets: list,
    optimum_run: int,
    initialisation_limit: int,
    plot_saving_path: str = None,
    **kwargs
) -> None:
    fig_size = kwargs.get('fig_size', (8, 6))
    init_color = '#22265A'
    opt_color  = '#F39103'
    side_color = '#28A745'
    side_color = kwargs.get('side_color', '#28A745')
    side_limit = kwargs.get('sideproduct_limit', 200)
    side_optimum = kwargs.get('side_optimum', None)

    data_scaled = data.copy()
    scaler = MinMaxScaler()
    data_scaled[continuous_cols] = scaler.fit_transform(data_scaled[continuous_cols])

    # Get the optimum run values.
    optimum_choices_cont = data_scaled.loc[optimum_run, continuous_cols]
    optimum_choices_cat = data.loc[optimum_run, categorical_cols]  # use original for categorical comparison

    # Calculate distances in the choices space.
    distances = []
    for idx, row in data_scaled.iterrows():
        # compute distance as before
        cont_distance = np.linalg.norm(row[continuous_cols] - optimum_choices_cont)
        cat_distance = sum(data.loc[idx, col] != optimum_choices_cat[col] for col in categorical_cols)
        distances.append(cont_distance + cat_distance)

    if side_optimum is not None:
        side_optimum = data_scaled.loc[side_optimum, continuous_cols]
        side_distances = []
        for idx, row in data_scaled.iterrows():
            cont_distance = np.linalg.norm(row[continuous_cols] - side_optimum)
            cat_distance = sum(data.loc[idx, col] != optimum_choices_cat[col] for col in categorical_cols)
            side_distances.append(cont_distance + cat_distance)
        distances = np.array(distances) + np.array(side_distances)

    run_numbers = np.arange(len(distances)) + 1
    expl_mask = run_numbers <= initialisation_limit
    side_mask = run_numbers > side_limit
    opt_mask  = ~(expl_mask | side_mask)


    num_targets = len(targets)
    xlabel = (
        r"$D_i = \left\| \mathbf{x}_i^{\mathrm{cont}} - "
        r"\mathbf{x}^{*\mathrm{cont}} \right\|_2 + \lambda \cdot N_{\mathrm{cat},i}$"
    )

    if num_targets == 1:
        target = targets[0]
        diffs = data[target].values
        opt_val = data.loc[optimum_run, target]

        plt.figure(figsize=fig_size)
        # Exploration
        plt.scatter(
            np.array(distances)[expl_mask],
            diffs[expl_mask],
            color=init_color, edgecolor='k',
            alpha=0.7, label='Initialization'
        )
        # Optimisation
        plt.scatter(
            np.array(distances)[opt_mask],
            diffs[opt_mask],
            color=opt_color, edgecolor='k',
            alpha=0.7, label='Optimisation '
        )
        # Sideproduct Runs
        if side_mask.any():
            plt.scatter(
                np.array(distances)[side_mask],
                diffs[side_mask],
                color=side_color, edgecolor='k',
                alpha=0.7, marker='^',
                label='Optimization 9b'
            )
        # Optimum line
        plt.axhline(opt_val, color='r', linestyle='--', label='Optimum')

        # labels & limits
        plt.xlabel(xlabel)
        plt.ylabel(target)
        # plt.title(f"{target} vs. Distance in Feature Space")

        if target in ['Yield [%]', 'Conversion [%]']:
            plt.ylim(0, 100)
        else:
            y_min = min(diffs) - 0.1 * abs(min(diffs))
            y_max = max(diffs) + 0.1 * abs(max(diffs))
            plt.ylim(y_min, y_max)

        plt.legend()
        if plot_saving_path:
            name_plus = kwargs.get('name_plus', '')
            plt.savefig(f"{plot_saving_path}/{name_plus}deviation_vs_distance_{target}2.png", dpi=1200)
            # plt.savefig(f"{plot_saving_path}/{name_plus}deviation_vs_distance_{target}.svg", dpi=1200)
        plt.show()

    else:
        fig, axes = plt.subplots(
            num_targets, 1,
            figsize=(fig_size[0], fig_size[1] * num_targets),
            sharex=True
        )
        if num_targets == 1:
            axes = [axes]

        for ax, target in zip(axes, targets):
            diffs = data[target].values
            opt_val = data.loc[optimum_run, target]

            ax.scatter(
                np.array(distances)[expl_mask],
                diffs[expl_mask],
                color=init_color, edgecolor='k',
                alpha=0.7, label='Exploration'
            )
            ax.scatter(
                np.array(distances)[opt_mask],
                diffs[opt_mask],
                color=opt_color, edgecolor='k',
                alpha=0.7, label='Optimisation '
            )
            if side_mask.any():
                ax.scatter(
                    np.array(distances)[side_mask],
                    diffs[side_mask],
                    color=side_color, edgecolor='k',
                    alpha=0.7, marker='^',
                    label='Optimization 9b'
                )

            ax.axhline(opt_val, color='r', linestyle='--', label='Optimum')
            ax.set_ylabel(target)

            if target in ['Yield [%]', 'Conversion [%]']:
                ax.set_ylim(0, 100)
            else:
                y_min = min(diffs) - 0.1 * abs(min(diffs))
                y_max = max(diffs) + 0.1 * abs(max(diffs))
                ax.set_ylim(y_min, y_max)

            ax.legend()

        axes[-1].set_xlabel(xlabel)
        # fig.suptitle("Difference in Targets vs. Feature‐Space Distance", fontsize=14)
        plt.tight_layout(rect=[0, 0, 1, 0.96])

        if plot_saving_path:
            name_plus = kwargs.get('name_plus', '')
            fig.savefig(f"{plot_saving_path}/{name_plus}deviation_vs_distance_multi.png", dpi=1200)
            # fig.savefig(f"{plot_saving_path}/{name_plus}deviation_vs_distance_multi.svg", dpi=1200)
        plt.show()

# print(data_1)
# print(data_2)
plot_deviation_vs_distance(data, colum_types['categorical'], colum_types['numerical'], targets=targets, optimum_run=optimum_run-1, initialisation_limit=exploration_limit, plot_saving_path=plot_saving_path,
                           # sideproduct_limit=sideprod_start-1
                           )
# plot_deviation_vs_distance(data_2, colum_types['categorical'], colum_types['numerical'], targets=targets, optimum_run=optimum_run_2-1, initialisation_limit=4, plot_saving_path=plot_saving_path,  name_plus = 'B')
# plot_radial_correlation(data, colum_types['categorical'], colum_types['numerical'], targets=targets)
# plot_radial_target_deviation(data,[], colum_types['numerical'], target=targets, optimum_run=optimum_run)



## Correlational Plotting
now that we plotted results and choiches separately, it is time to plot and calculate fun things such as correlation coefficients and see what data likes playing with what other data.

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable


def plot_correlation_heatmap(data: pd.DataFrame, categorical_cols: List[str], continuous_cols: List[str],
                             targets: List[str], plot_saving_path: Optional[str] = None, **kwargs) -> None:
    """
    Plots normalized correlation heatmaps for continuous-continuous and categorical-continuous relationships, with:
    - Pearson/eta-squared correlations in one heatmap.
    - Spearman/omega-squared correlations in the other heatmap.
    - Min-max scaling applied to continuous columns for uniform comparison.

    Parameters:
    data: pd.DataFrame - The DataFrame containing the data to analyze.
    categorical_cols: List[str] - List of column names with categorical variables.
    continuous_cols: List[str] - List of column names with continuous variables.
    targets: List[str] - List of target column names (always treated as continuous).
    plot_saving_path: str, optional - Path to save the plot image. If None, the plot is not saved.

    Additional kwargs:
    fig_size: Tuple[int, int] - Custom figure size for each heatmap, default is (10, 8).
    cmap: str - Name of the colormap to use for the heatmap, default is 'viridis'.
    nan_color: str - Color to use for NaN values, default is 'lightgray'.

    The function displays two side-by-side heatmaps: one for Pearson/eta-squared values and one for Spearman/omega-squared values.
    """

    fig_size = kwargs.get('fig_size', (10, 8))
    cmap = kwargs.get('cmap', default_sequential_cmap)
    cmap = load_cmap(cmap, cmap_type='continuous')  # Check if cmap is valid
    # Treat targets as continuous columns
    continuous_cols = list(set(continuous_cols + targets))

    # Scale continuous variables for uniformity
    data_scaled = data.copy()
    scaler = MinMaxScaler()
    data_scaled[continuous_cols] = scaler.fit_transform(data_scaled[continuous_cols])

    # Create a unique list of all columns to include in the heatmap
    all_cols = continuous_cols + categorical_cols

    # Initialize the correlation matrices
    lower_corr_matrix = pd.DataFrame(index=all_cols, columns=all_cols, dtype=float)
    upper_corr_matrix = pd.DataFrame(index=all_cols, columns=all_cols, dtype=float)

    # Compute correlations and effect sizes
    for col1 in all_cols:
        for col2 in all_cols:
            if col1 == col2:
                lower_corr_matrix.loc[col1, col2] = 1.0  # Self-correlation
                upper_corr_matrix.loc[col1, col2] = 1.0
            elif col1 in continuous_cols and col2 in continuous_cols:
                # Lower triangle - Pearson, Upper triangle - Spearman
                lower_corr_matrix.loc[col1, col2] = pearsonr(data_scaled[col1], data_scaled[col2])[0]  # Pearson
                upper_corr_matrix.loc[col2, col1] = spearmanr(data_scaled[col1], data_scaled[col2])[0]  # Spearman
            elif col1 in categorical_cols and col2 in continuous_cols:
                # Lower triangle - eta-squared, Upper triangle - omega-squared
                total_ss = np.sum((data_scaled[col2] - data_scaled[col2].mean()) ** 2)
                ss_between = sum([
                    len(group) * (group.mean() - data_scaled[col2].mean()) ** 2
                    for _, group in data_scaled.groupby(col1)[col2]
                ])
                eta_squared = ss_between / total_ss
                lower_corr_matrix.at[col1, col2] = max(0,eta_squared)

                ss_within = total_ss - ss_between
                omega_squared = (ss_between - (len(data_scaled.groupby(col1)) - 1) * ss_within /
                                 (len(data_scaled) - len(data_scaled.groupby(col1)))) / total_ss
                upper_corr_matrix.at[col2, col1] = max(0,omega_squared)
            elif col1 in categorical_cols and col2 in categorical_cols:
                # Categorical-categorical relationships: Set to NaN
                lower_corr_matrix.loc[col1, col2] = np.nan
                upper_corr_matrix.loc[col1, col2] = np.nan

    # Fill NaNs with a placeholder for visualization
    lower_corr_matrix = lower_corr_matrix.fillna(np.nan)
    upper_corr_matrix = upper_corr_matrix.fillna(np.nan)

    # Create a mask for the upper and lower triangles
    mask_lower = np.triu(np.ones_like(lower_corr_matrix, dtype=bool))
    mask_upper = np.tril(np.ones_like(upper_corr_matrix, dtype=bool))

    # Plot side-by-side heatmaps
    fig, axes = plt.subplots(2, 1, figsize=(fig_size[0], 2 * fig_size[1]))

    # Pearson/eta-squared heatmap with NaN color
    sns.heatmap(lower_corr_matrix, mask=mask_lower, annot=True, cmap=cmap, fmt=".2f",
                cbar=True, ax=axes[0], square=True, linewidths=0.5,
                cbar_kws={'label': 'Pearson'})
    axes[0].set_title(r"Pearson (Continuous)")

    # Spearman/omega-squared heatmap with NaN color
    sns.heatmap(upper_corr_matrix, mask=mask_upper, annot=True, cmap=cmap, fmt=".2f",
                cbar=True, ax=axes[1], square=True, linewidths=0.5,
                cbar_kws={'label': 'Spearman'})
    axes[1].set_title(r"Spearman (Continuous)")

    # Adjust layout and save if path is provided
    plt.tight_layout()
    if plot_saving_path is not None:
        plt.savefig(f"{plot_saving_path}/correlation_heatmap_side_by_side.png")
    plt.show()

def plot_correlation_heatmap2(data: pd.DataFrame, categorical_cols: List[str], continuous_cols: List[str],
                             targets: List[str], plot_saving_path: Optional[str] = None, **kwargs) -> None:
    """
    Plots normalized correlation heatmaps for continuous-continuous and categorical-continuous relationships.
    """
    fig_size = kwargs.get('fig_size', (10, 8))
    cmap_continuous = kwargs.get('cmap_continuous', default_sequential_cmap)  # default cmap for continuous-continuous
    cmap_cont = load_cmap(cmap_continuous, cmap_type='continuous')  # Check if cmap is valid
    cmap_categorical = kwargs.get('cmap_categorical', default_sequential_cmap2)  # second cmap for categorical-continuous
    cmap_cat = load_cmap(cmap_categorical, cmap_type='continuous')  # Check if cmap is valid
    nan_color = kwargs.get('nan_color', 'lightgray')

    # Treat targets as continuous columns
    continuous_cols = list(set(continuous_cols + targets))

    # Scale continuous variables for uniformity
    data_scaled = data.copy()
    scaler = MinMaxScaler()
    data_scaled[continuous_cols] = scaler.fit_transform(data_scaled[continuous_cols])

    # Create a unique list of all columns to include in the heatmap
    all_cols = continuous_cols + categorical_cols

    # Initialize the correlation matrices
    pearson_eta = pd.DataFrame(index=all_cols, columns=all_cols, dtype=float)
    spearman_omega = pd.DataFrame(index=all_cols, columns=all_cols, dtype=float)

    # Compute correlations and effect sizes
    for col1 in all_cols:
        for col2 in all_cols:
            if col1 == col2:
                pearson_eta.loc[col1, col2] =  1.0 # Self-correlation
                spearman_omega.loc[col1, col2] = 1.0
            elif col1 in continuous_cols and col2 in continuous_cols:
                # Pearson and Spearman for continuous-continuous relationships
                pearson_eta.loc[col1, col2] = pearsonr(data_scaled[col1], data_scaled[col2])[0]  # Pearson
                spearman_omega.loc[col2, col1] = spearmanr(data_scaled[col1], data_scaled[col2])[0]  # Spearman
            elif col1 in categorical_cols and col2 in continuous_cols:
                # Eta-squared and Omega-squared for categorical-continuous relationships
                total_ss = np.sum((data_scaled[col2] - data_scaled[col2].mean()) ** 2)
                ss_between = sum([
                    len(group) * (group.mean() - data_scaled[col2].mean()) ** 2
                    for _, group in data_scaled.groupby(col1)[col2]
                ])
                eta_squared = ss_between / total_ss
                pearson_eta.at[col1, col2] = max(0,eta_squared)

                ss_within = total_ss - ss_between
                omega_squared = (ss_between - (len(data_scaled.groupby(col1)) - 1) * ss_within /
                                 (len(data_scaled) - len(data_scaled.groupby(col1)))) / total_ss
                spearman_omega.at[col2, col1] = max(0,omega_squared)
            elif col1 in categorical_cols and col2 in categorical_cols:
                # Set NaN for categorical-categorical relationships
                pearson_eta.loc[col1, col2] = np.nan
                spearman_omega.loc[col1, col2] = np.nan

    # Masks for separating continuous-continuous and categorical-continuous values
    pearson_eta = pearson_eta.fillna(np.nan)
    spearman_omega = spearman_omega.fillna(np.nan)


    # Create masks for continuous-continuous and categorical-continuous values
    mask_continuous = pearson_eta.notna() & spearman_omega.notna()
    mask_categorical = ~mask_continuous
    mask_lower = np.triu(np.ones_like(pearson_eta, dtype=bool))
    mask_upper = np.tril(np.ones_like(spearman_omega, dtype=bool))

    # Plot side-by-side heatmaps
    fig, axes = plt.subplots(2, 1, figsize=(fig_size[0], 2 * fig_size[1]))

    #categoricals

    sns.heatmap(pearson_eta, mask=mask_categorical|mask_lower, annot=True, cmap=cmap_cont, fmt=".2f",
                cbar=True, ax=axes[0], square=True, linewidths=0.5,
                cbar_kws={'label': 'Pearson'})
    sns.heatmap(spearman_omega, mask=mask_categorical|mask_upper, annot=True, cmap=cmap_cont, fmt=".2f",
                cbar=True, ax=axes[1], square=True, linewidths=0.5,
                cbar_kws={'label': 'Spearman'})

    # Continous
    sns.heatmap(pearson_eta, mask=mask_continuous|mask_lower, annot=True, cmap=cmap_cat, fmt=".2f",
                cbar=True, ax=axes[0], square=True, linewidths=0.5,
                cbar_kws={'label': r'$\eta^2$'})
    sns.heatmap(spearman_omega, mask=mask_continuous|mask_upper, annot=True, cmap=cmap_cat, fmt=".2f",
                cbar=True, ax=axes[1], square=True, linewidths=0.5,
                cbar_kws={'label': r'$\omega^2$'})
    axes[1].set_title(r"Spearman (Continuous) and $\omega^2$ (Categorical)")
    axes[0].set_title(r"Pearson (Continuous) and $\eta^2$ (Categorical)")

    # Adjust layout and save if path is provided
    plt.tight_layout()
    if plot_saving_path is not None:
        plt.savefig(f"{plot_saving_path}/correlation_heatmap_side_by_side_2.png")
    plt.show()


def plot_correlation_heatmap_with_dummies(data: pd.DataFrame, categorical_cols: List[str], continuous_cols: List[str],
                                          targets: List[str], plot_saving_path: Optional[str] = None, **kwargs) -> None:
    """
    Plots normalized correlation heatmaps for continuous-continuous and categorical-continuous relationships, with:
    - Pearson/eta-squared correlations in one heatmap.
    - Spearman/omega-squared correlations in the other heatmap.
    - Min-max scaling applied to all columns for uniform comparison.

    Parameters:
    data: pd.DataFrame - The DataFrame containing the data to analyze.
    categorical_cols: List[str] - List of column names with categorical variables.
    continuous_cols: List[str] - List of column names with continuous variables.
    targets: List[str] - List of target column names (always treated as continuous).
    plot_saving_path: str, optional - Path to save the plot image. If None, the plot is not saved.

    Additional kwargs:
    fig_size: Tuple[int, int] - Custom figure size for each heatmap, default is (10, 8).
    cmap: str - Name of the colormap to use for the heatmap, default is 'viridis'.
    nan_color: str - Color to use for NaN values, default is 'lightgray'.

    The function displays two side-by-side heatmaps: one for Pearson/eta-squared values and one for Spearman/omega-squared values.
    """

    fig_size = kwargs.get('fig_size', (10, 8))
    cmap = kwargs.get('cmap', default_sequential_cmap)
    cmap = load_cmap(cmap, cmap_type='continuous')  # Check if cmap is valid
    # Treat targets as continuous columns and apply dummy encoding to categorical columns
    continuous_cols = list(set(continuous_cols + targets))
    data_with_dummies = pd.get_dummies(data, columns=categorical_cols, drop_first=False)

    # Scale all continuous and dummy columns for uniformity
    scaler = MinMaxScaler()
    data_with_dummies[continuous_cols + [col for col in data_with_dummies.columns if col not in continuous_cols]] = \
        scaler.fit_transform(data_with_dummies[continuous_cols + [col for col in data_with_dummies.columns if col not in continuous_cols]])

    # Create a unique list of all columns for correlation
    all_cols = continuous_cols + [col for col in data_with_dummies.columns if col not in continuous_cols]

    # Initialize the correlation matrices
    lower_corr_matrix = pd.DataFrame(index=all_cols, columns=all_cols, dtype=float)
    upper_corr_matrix = pd.DataFrame(index=all_cols, columns=all_cols, dtype=float)

    # Compute correlations
    for col1 in all_cols:
        for col2 in all_cols:
            if col1 == col2:
                lower_corr_matrix.loc[col1, col2] = 1.0  # Self-correlation
                upper_corr_matrix.loc[col1, col2] = 1.0
            elif col1 in continuous_cols and col2 in continuous_cols:
                # Lower triangle - Pearson, Upper triangle - Spearman
                lower_corr_matrix.loc[col1, col2] = pearsonr(data_with_dummies[col1], data_with_dummies[col2])[0]  # Pearson
                upper_corr_matrix.loc[col2, col1] = spearmanr(data_with_dummies[col1], data_with_dummies[col2])[0]  # Spearman
            elif col1 in data_with_dummies.columns and col2 in continuous_cols:
                # Lower triangle - eta-squared, Upper triangle - omega-squared
                total_ss = np.sum((data_with_dummies[col2] - data_with_dummies[col2].mean()) ** 2)
                ss_between = sum([
                    len(group) * (group.mean() - data_with_dummies[col2].mean()) ** 2
                    for _, group in data_with_dummies.groupby(col1)[col2]
                ])
                eta_squared = ss_between / total_ss
                lower_corr_matrix.at[col1, col2] = max(0,eta_squared)

                ss_within = total_ss - ss_between
                omega_squared = (ss_between - (len(data_with_dummies.groupby(col1)) - 1) * ss_within /
                                 (len(data_with_dummies) - len(data_with_dummies.groupby(col1)))) / total_ss
                upper_corr_matrix.at[col2, col1] = max(0,omega_squared)

    # Fill NaNs with a placeholder for visualization
    lower_corr_matrix = lower_corr_matrix.fillna(np.nan)
    upper_corr_matrix = upper_corr_matrix.fillna(np.nan)

    # Create a mask for the upper and lower triangles
    mask_lower = np.triu(np.ones_like(lower_corr_matrix, dtype=bool))
    mask_upper = np.tril(np.ones_like(upper_corr_matrix, dtype=bool))

    # Plot side-by-side heatmaps
    fig, axes = plt.subplots(2, 1, figsize=(fig_size[0], 2 * fig_size[1]))

    # Pearson/eta-squared heatmap with NaN color
    sns.heatmap(lower_corr_matrix, mask=mask_lower, annot=True, cmap=cmap, fmt=".2f",
                cbar=True, ax=axes[0], square=True, linewidths=0.5,
                cbar_kws={'label': 'Pearson/eta²'}, annot_kws={'fontsize':8})
    axes[0].set_title(r"Pearson (Continuous) and $\eta^2$ (Categorical)")

    # Spearman/omega-squared heatmap with NaN color
    sns.heatmap(upper_corr_matrix, mask=mask_upper, annot=True, cmap=cmap, fmt=".2f",
                cbar=True, ax=axes[1], square=True, linewidths=0.5,
                cbar_kws={'label': 'Spearman/omega²'}, annot_kws={'fontsize':8})
    axes[1].set_title(r"Spearman (Continuous) and $\omega^2$ (Categorical)")

    # Adjust layout and save if path is provided
    plt.tight_layout()
    if plot_saving_path is not None:
        plt.savefig(f"{plot_saving_path}/correlation_heatmap_side_by_side_dummies.png")
    plt.show()


def plot_correlation_heatmap_with_dummies2(data: pd.DataFrame, categorical_cols: List[str], continuous_cols: List[str],
                                          targets: List[str], plot_saving_path: Optional[str] = None, **kwargs) -> None:
    """
    Plots normalized correlation heatmaps for continuous-continuous and categorical-continuous relationships,
    using Pearson/eta-squared correlations in one heatmap and Spearman/omega-squared correlations in the other.
    """
    fig_size = kwargs.get('fig_size', (10, 8))
    cmap_continuous = kwargs.get('cmap_continuous', default_sequential_cmap)
    cmap_cont = load_cmap(cmap_continuous, cmap_type='continuous')
    cmap_categorical = kwargs.get('cmap_categorical', default_sequential_cmap2)
    cmap_cat = load_cmap(cmap_categorical, cmap_type='continuous')

    # treat targets as continuous
    continuous_cols = list(set(continuous_cols + targets))
    # dummy encode categoricals
    data_dum = pd.get_dummies(data, columns=categorical_cols, drop_first=False, prefix_sep=r'$\rightarrow$')
    # scale all continuous for uniformity
    scaler = MinMaxScaler()
    data_dum[continuous_cols] = scaler.fit_transform(data_dum[continuous_cols])

    # build full column list: continuous + dummies
    all_cols = continuous_cols + [c for c in data_dum.columns if c not in continuous_cols]

    # init matrices
    pearson_eta = pd.DataFrame(index=all_cols, columns=all_cols, dtype=float)
    spearman_omega = pd.DataFrame(index=all_cols, columns=all_cols, dtype=float)

    # compute
    for c1 in all_cols:
        for c2 in all_cols:
            if c1 == c2:
                pearson_eta.loc[c1, c2] = 1.0
                spearman_omega.loc[c1, c2] = 1.0
            elif c1 in continuous_cols and c2 in continuous_cols:
                pearson_eta.loc[c1, c2] = pearsonr(data_dum[c1], data_dum[c2])[0]
                spearman_omega.loc[c2, c1] = spearmanr(data_dum[c1], data_dum[c2])[0]
            else:
                # categorical-continuous pairs only
                if c1 in data_dum.columns and c2 in continuous_cols and c1 not in continuous_cols:
                    total_ss = ((data_dum[c2] - data_dum[c2].mean())**2).sum()
                    ss_between = sum(
                        len(g)*((g.mean()-data_dum[c2].mean())**2)
                        for _, g in data_dum.groupby(c1)[c2]
                    )
                    eta2 = ss_between / total_ss
                    pearson_eta.at[c1, c2] = max(0, eta2)
                    ss_within = total_ss - ss_between
                    omega2 = (
                        ss_between - (len(data_dum.groupby(c1))-1)*ss_within/
                        (len(data_dum)-len(data_dum.groupby(c1)))
                    ) / total_ss
                    spearman_omega.at[c2, c1] = max(0, omega2)
                else:
                    pearson_eta.loc[c1, c2] = np.nan
                    spearman_omega.loc[c1, c2] = np.nan

    # masks
    mask_cont = pearson_eta.notna() & spearman_omega.notna()
    mask_cat = ~mask_cont
    mask_lower = np.triu(np.ones_like(pearson_eta, dtype=bool))
    mask_upper = np.tril(np.ones_like(spearman_omega, dtype=bool))

    # plot
    fig, axes = plt.subplots(2, 1, figsize=(fig_size[0], 2*fig_size[1]))

    # Pearson / eta²
    sns.heatmap(
        pearson_eta,
        mask=(mask_cat|mask_lower),
        annot=True, fmt=".2f",
        cmap=cmap_cont, cbar=True,
        ax=axes[0], square=True, linewidths=0.5,
        cbar_kws={'label': 'Pearson'}
    )
    sns.heatmap(
        pearson_eta,
        mask=(mask_cont|mask_lower),
        annot=True, fmt=".2f",
        cmap=cmap_cat, cbar=True,
        ax=axes[0], square=True, linewidths=0.5,
        cbar_kws={'label': r'$\eta^2$'}
    )
    axes[0].set_title(r"Pearson (Continuous) and $\eta^2$ (Categorical)")

    # Spearman / omega²
    sns.heatmap(
        spearman_omega,
        mask=(mask_cat|mask_upper),
        annot=True, fmt=".2f",
        cmap=cmap_cont, cbar=True,
        ax=axes[1], square=True, linewidths=0.5,
        cbar_kws={'label': 'Spearman'}
    )
    sns.heatmap(
        spearman_omega,
        mask=(mask_cont|mask_upper),
        annot=True, fmt=".2f",
        cmap=cmap_cat, cbar=True,
        ax=axes[1], square=True, linewidths=0.5,
        cbar_kws={'label': r'$\omega^2$'}
    )
    axes[1].set_title(r"Spearman (Continuous) and $\omega^2$ (Categorical)")

    plt.tight_layout()
    if plot_saving_path:
        name_plus = kwargs.get('name_plus', '')
        fig.savefig(f"{plot_saving_path}/{name_plus}correlation_heatmap_side_by_side_dummies_2.png", dpi=1200)
        fig.savefig(f"{plot_saving_path}/{name_plus}correlation_heatmap_side_by_side_dummies_2.svg",dpi=1200)

    plt.show()




def calculate_omega_squared(df, target, combined_values, count_k):
    """Calculate omega-squared for a specific combined value of categorical variables on a continuous target."""
    total_ss = np.sum((df[target] - df[target].mean()) ** 2)
    group = df.loc[combined_values, target]
    ss_between = len(group) * (group.mean() - df[target].mean()) ** 2
    n = len(df)
    k = count_k
    ss_within = total_ss - ss_between
    omega_squared = (ss_between - (k - 1) * ss_within / (n - k)) / total_ss if total_ss != 0 else 0
    omega_squared = max(0, omega_squared)
    return omega_squared

def plot_categorical_combination_correlation(data: pd.DataFrame, categorical_cols: List[str], targets: List[str],
                                             plot_saving_path: Optional[str] = None, **kwargs) -> None:
    """
    Plots η² and ω² heatmaps for combinations of categorical columns against each target in a grid layout.

    Each target has a 3x3 grid, where the lower triangle contains η² heatmaps, and the upper triangle contains ω² heatmaps.
    Each heatmap represents the effect of combinations of categorical values on the target.

    Parameters:
    data (pd.DataFrame): DataFrame with data to analyze.
    categorical_cols (List[str]): List of categorical columns.
    targets (List[str]): List of target column names (continuous).
    plot_saving_path (Optional[str]): Path to save the plot image, if provided.

    Additional kwargs:
    fig_size: Tuple[int, int] - Custom figure size for each heatmap, default is (10, 8).
    cmap: str - Colormap for the heatmap, default is 'viridis'.

    Returns:
    None
    """
    if not categorical_cols or len(categorical_cols) == 0:
        print("No categorical columns found in the data.")
        return
    fig_size = kwargs.get('fig_size', (10, 8))
    cmap_name = kwargs.get('cmap', default_sequential_cmap)
    cmap = load_cmap(cmap_name, cmap_type='continuous')  # Check if cmap is valid

    # Filter out categorical columns with only one unique value
    valid_categorical_cols = [col for col in categorical_cols if data[col].nunique() > 1]

    # Get all pairs of categorical columns
    cat_combinations = list(combinations(valid_categorical_cols, 2))

    # Create a separate figure for each target
    for target in targets:
        # Create a grid for each target
        n = len(valid_categorical_cols)
        fig, axes = plt.subplots(n, n, figsize=(fig_size[0], fig_size[1]))
        # fig.suptitle(f"Effect of Categorical Pairs on Target: {target}", fontsize=16, y=1.02)

        for i, cat1 in enumerate(valid_categorical_cols):
            for j, cat2 in enumerate(valid_categorical_cols):
                if i == j:
                    axes[i, j].axis("off")  # Leave the diagonal blank
                elif i > j:
                    # Lower triangle: η² heatmaps
                    unique_cat1 = data[cat1].unique()
                    unique_cat2 = data[cat2].unique()
                    eta_matrix = pd.DataFrame(index=unique_cat1, columns=unique_cat2, dtype=float)

                    for val1 in unique_cat1:
                        for val2 in unique_cat2:
                            subset = data[(data[cat1] == val1) & (data[cat2] == val2)]
                            if not subset.empty:
                                eta_matrix.loc[val1, val2] = calculate_eta_squared(data, target, subset.index)

                    sns.heatmap(eta_matrix, annot=True, cmap=cmap, cbar=True, ax=axes[i, j], vmin=0, vmax=1, fmt=".2f")
                    axes[i, j].set_title(f"{cat1} vs {cat2} ({r'$\eta^2$'})", fontsize=10)
                    axes[i, j].set_xlabel(cat2)
                    axes[i, j].set_ylabel(cat1)

                elif i < j:
                    # Upper triangle: ω² heatmaps
                    unique_cat1 = data[cat1].unique()
                    unique_cat2 = data[cat2].unique()
                    omega_matrix = pd.DataFrame(index=unique_cat1, columns=unique_cat2, dtype=float)
                    count_k = data[[cat1, cat2]].drop_duplicates().shape[0]
                    for val1 in unique_cat1:
                        for val2 in unique_cat2:
                            subset = data[(data[cat1] == val1) & (data[cat2] == val2)]
                            if not subset.empty:
                                omega_matrix.loc[val1, val2] = calculate_omega_squared(data, target, subset.index, count_k)

                    sns.heatmap(omega_matrix, annot= True, cmap=cmap, cbar=True, ax=axes[i, j], vmin=0, vmax=1, fmt=".2f")
                    axes[i, j].set_title(f"{cat1} vs {cat2} ({r'$\omega^2$'})", fontsize=10)
                    axes[i, j].set_xlabel(cat2)
                    axes[i, j].set_ylabel(cat1)

        # cbar = fig.colorbar(ScalarMappable(cmap=cmap), ax = axes)
        # cbar.set_label('Effect Size', rotation=270, labelpad=15)

        plt.tight_layout()
        if plot_saving_path:
            plt.savefig(f"{plot_saving_path}/categorical_combination_correlation_{target}.png")
        plt.show()




In [ ]:

# plot_correlation_heatmap(data, colum_types['categorical'], colum_types['numerical'], targets, plot_saving_path=plot_saving_path)
# plot_correlation_heatmap2(data, colum_types['categorical'], colum_types['numerical'], targets, plot_saving_path=plot_saving_path)
# plot_correlation_heatmap_with_dummies(data, colum_types['categorical'], colum_types['numerical'], targets, plot_saving_path=plot_saving_path)
plot_correlation_heatmap_with_dummies2(data, colum_types['categorical'], colum_types['numerical'], targets, plot_saving_path=plot_saving_path, fig_size=(12, 10), together = False)

# plot_correlation_heatmap_with_dummies2(data_2, colum_types['categorical'], colum_types['numerical'], targets, plot_saving_path=plot_saving_path, fig_size=(12, 10), together = False, name_plus = 'data_2')
# plot_categorical_combination_correlation(data_, colum_types['categorical'], targets, plot_saving_path=plot_saving_path, fig_size=(20, 20))

In [ ]:

def omega_squared(data: pd.DataFrame, cat: str, target: str) -> float:
    """
    Computes ω² (omega-squared) for a categorical→continuous relationship.
    Formula: (SS_between - (k-1)*MS_within) / (SS_total + MS_within)
    """
    y = data[target]
    groups = [g[target] for _, g in data.groupby(cat)]
    n = len(data)
    k = len(groups)
    grand_mean = y.mean()

    # total sum of squares
    ss_total = ((y - grand_mean) ** 2).sum()
    # between
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
    # within
    ss_within = sum(((g - g.mean()) ** 2).sum() for g in groups)
    # mean square within
    ms_within = ss_within / (n - k) if n != k else 0

    num = ss_between - (k - 1) * ms_within
    den = ss_total + ms_within
    return max(0.0, num / den) if den != 0 else 0.0

def feature_target_scores(
    df: pd.DataFrame,
    continuous_cols: list,
    categorical_cols: list,
    target_col: str
) -> pd.DataFrame:
    """
    Returns a DataFrame indexed by feature name with columns:
      - 'pearson', 'spearman'      (for continuous features)
      - 'eta2',    'omega2'        (for categorical features)
    Others are NaN.
    """
    records = []
    for feat in continuous_cols:
        y, x = df[target_col].values, df[feat].values
        pear, _ = pearsonr(x, y)
        spea, _ = spearmanr(x, y)
        records.append({
            'feature': feat,
            'pearson': pear,
            'spearman': spea,
            'eta2':    np.nan,
            'omega2':  np.nan
        })

    for feat in categorical_cols:
        eta2 = compute_eta_squared(df, feat, target_col)
        om2  = omega_squared(df, feat, target_col)
        records.append({
            'feature': feat,
            'pearson': np.nan,
            'spearman': np.nan,
            'eta2':    eta2,
            'omega2':  om2
        })

    return pd.DataFrame(records).set_index('feature')

def compare_feature_profiles(
    df1: pd.DataFrame, df2: pd.DataFrame,
    continuous_cols: list, categorical_cols: list,
    target_col: str
):
    """
    Computes and aligns feature→target scores for two tasks, then
    returns a dict of inter-task correlations & norms for each metric.
    """
    s1 = feature_target_scores(df1, continuous_cols, categorical_cols, target_col)
    s2 = feature_target_scores(df2, continuous_cols, categorical_cols, target_col)

    # align on common features
    merged = s1.join(s2, lsuffix='_1', rsuffix='_2', how='inner')

    stats = {}
    for metric in ['pearson', 'spearman', 'eta2', 'omega2']:
        col1, col2 = metric + '_1', metric + '_2'
        valid = merged[[col1, col2]].dropna()
        if not valid.empty:
            r = valid[col1].corr(valid[col2], method='pearson')
            diff_norm = np.linalg.norm(valid[col1] - valid[col2])
        else:
            r, diff_norm = np.nan, np.nan
        stats[metric] = {'r': r, 'diff_norm': diff_norm}

    return merged, stats

merged_scores, stats = compare_feature_profiles(
    data_1, data_2,
    continuous_cols=colum_types['numerical'],
    categorical_cols=colum_types['categorical'],
    target_col=targets[0]
)

# 2) Inspect numeric results
for metric in stats:
    print(f"{metric}: correlation = {stats[metric]['r']:.3f}, "
          f"||Δ|| = {stats[metric]['diff_norm']:.3f}")

# 3) (Optional) Visualize pearson‐profile agreement
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(
    merged_scores['pearson_1'],
    merged_scores['pearson_2'],
    s=60, edgecolor='k', alpha=0.7
)
lims = [
    min(ax.get_xlim()[0], ax.get_ylim()[0]),
    max(ax.get_xlim()[1], ax.get_ylim()[1])
]
ax.plot(lims, lims, '--', color='gray')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Pearson (task 1)')
ax.set_ylabel('Pearson (task 2)')
ax.set_title(f"Pearson‐profile r = {stats['pearson']['r']:.2f}")
plt.tight_layout()
plt.show()




# 1) Melt into long form
df = merged_scores.reset_index().melt(
    id_vars=['feature'],
    value_vars=merged_scores.columns,
    var_name='metric_task',
    value_name='value'
)

# 2) Split metric_task into two new columns
df[['metric','task']] = df['metric_task'].str.rsplit('_', n=1, expand=True)
# e.g. 'pearson_1' → metric='pearson', task='1'
palette = sns.color_palette("ninetales", n_colors=len(df['metric'].unique()))

# 3) Plot
plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=df,
    x='feature',        # categorical axis
    y='value',
    hue='metric',       # color by metric
    style='task',       # marker shape by task (1 vs 2)
    s=150,              # point size
    # palette='tab10',    # or any other palette
    markers={'1':'o','2':'^'},
    edgecolor='k',
    palette = palette,
    alpha=0.8
)

# 4) Polish
plt.axhline(0, color='gray', linestyle='--', linewidth=1)         # zero-reference
plt.xticks(rotation=45, ha='right')                               # rotate labels
plt.xlabel("Feature")
plt.ylabel("Score")
# plt.title("Feature→Target Profiles: Task 1 vs Task 2 across Metrics")
plt.legend(title="Metric / Task", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
if plot_saving_path is not None:
    plt.savefig(f'{plot_saving_path}/feature_target_scores.png', dpi=1200)
    plt.savefig(f'{plot_saving_path}/feature_target_scores.svg', dpi=1200)
plt.show()